# Authorized native-resolution crop fine-tuning
Frozen ResNet backbone; original training split only; 2-hour training cap, at most 3000 updates.


In [2]:
import os, sys, time, json, shutil, subprocess
from pathlib import Path

RUN_STARTED = time.monotonic()
import zipfile
archives = list(Path('/kaggle/input').rglob('solar_payload.bin'))
assert len(archives) == 1, archives
ASSETS = Path('/kaggle/working/solar_assets')
ASSETS.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    archive.extractall(ASSETS)
INPUTS = ASSETS
RUNTIME = Path('/kaggle/working/solar_m2f_runtime')
RUNTIME.mkdir(exist_ok=True)
for name in ['Mask2Former', 'detectron2']:
    target = RUNTIME / name
    if not target.exists():
        shutil.copytree(ASSETS / 'vendor' / name, target)
print('ASSETS:', ASSETS, flush=True)
print('Python:', sys.version, flush=True)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, flush=True)
assert torch.cuda.is_available(), 'A free Kaggle GPU is required; no CPU full run.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ninja', 'pycocotools', 'timm==0.9.16', 'fvcore==0.1.5.post20221221',
    'iopath==0.1.9', 'yacs', 'omegaconf>=2.1,<2.4', 'hydra-core',
    'termcolor', 'tabulate', 'cloudpickle', 'tensorboard', 'shapely',
    'scipy', 'submitit', 'black', 'packaging'], check=True, timeout=900)
# The Mask2Former R50 path needs no Detectron2 CUDA-specific operators.
# Build its normal CPU extension; deformable attention uses the existing
# project's differentiable PyTorch reference implementation during inference.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', MAX_JOBS='2')
log_path = Path('/kaggle/working/detectron2_install.log')
print('Building Detectron2 CPU extension; log:', log_path, flush=True)
with log_path.open('w') as log:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install',
        '--no-build-isolation', '--no-deps', str(RUNTIME / 'detectron2')],
        env=env, stdout=log, stderr=subprocess.STDOUT, timeout=1800)
print(log_path.read_text()[-8000:], flush=True)
assert result.returncode == 0, 'See detectron2_install.log'
sys.path[:0] = [str(ASSETS), str(RUNTIME / 'Mask2Former')]
from solar_runtime import install_msda_backend
BACKEND = install_msda_backend('torch')
import detectron2, mask2former
print('Runtime ready:', detectron2.__version__, BACKEND, flush=True)
print('SETUP_SECONDS:', round(time.monotonic() - RUN_STARTED), flush=True)

import csv, gc, hashlib, itertools, json, time, zipfile
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from pycocotools import mask as mu
from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.modeling import build_model
from detectron2.data import detection_utils as utils
from mask2former import add_maskformer2_config
from solar_inference import install_chunked_instance_inference
from solar_evaluation import mask_rle, pq_counts, summarize_counts

OUT = Path('/kaggle/working/calibration_results')
OUT.mkdir(exist_ok=True)
LIMIT_SECONDS = 5.5 * 3600
torch.set_grad_enabled(False)
torch.manual_seed(42)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
provenance = json.loads((ASSETS / 'provenance.json').read_text())
weight = ASSETS / 'model_best.pth'
assert hashlib.sha256(weight.read_bytes()).hexdigest() == provenance['checkpoint_sha256']
cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
cfg.set_new_allowed(True)
cfg.merge_from_file(str(ASSETS / 'config.yaml'))
cfg.MODEL.DEVICE = 'cuda'
cfg.DATASETS.TRAIN = ('solar_calibration_metadata',)
cfg.DATASETS.TEST = ()
model = build_model(cfg)
checkpoint = torch.load(weight, map_location='cpu', weights_only=True)
assert checkpoint['iteration'] == 36999
model.load_state_dict(checkpoint['model'], strict=True)
del checkpoint
gc.collect()
model.requires_grad_(False).eval()
install_chunked_instance_inference(model, .15)
records = json.loads((ASSETS / 'validation_records.json').read_text())
dates = sorted({r['file_name'][:8] for r in records},
               key=lambda x: hashlib.sha256(('calibration-v1-' + x).encode()).hexdigest())
cal_dates = set(dates[:len(dates)//2])
part = {r['file_name']: ('calibration' if r['file_name'][:8] in cal_dates else 'holdout') for r in records}
assert {r['file_name'][:8] for r in records if part[r['file_name']]=='calibration'}.isdisjoint(
       {r['file_name'][:8] for r in records if part[r['file_name']]=='holdout'})
(OUT/'split.json').write_text(json.dumps(part, indent=2))
root = next(ASSETS.rglob('test_images')).parent.parent
train_dir, test_dir = root/'train/train_images', root/'test/test_images'
raw = json.loads((root/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
gt = defaultdict(list)
for a in raw['annotations']:
    gt[a['image_id']].append(a)
VIEWS = ('native','hflip','vflip','rot180')

def encode(mask):
    r = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    r['counts'] = r['counts'].decode('ascii')
    return r

def candidates(path, view):
    assert time.monotonic()-RUN_STARTED < LIMIT_SECONDS, 'Calibration time cap reached'
    assert not model.training and not torch.is_grad_enabled()
    image = utils.read_image(str(path), format=cfg.INPUT.FORMAT)
    assert image.shape[:2] == (2048,2048)
    if view in ('hflip','rot180'): image = image[:,::-1]
    if view in ('vflip','rot180'): image = image[::-1]
    x = torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
    with torch.inference_mode():
        instances = model([{'image':x,'height':2048,'width':2048}])[0]['instances']
    result = []
    for idx in instances.scores.argsort(descending=True).tolist():
        mask = instances.pred_masks[idx].numpy()
        if view in ('hflip','rot180'): mask = mask[:,::-1]
        if view in ('vflip','rot180'): mask = mask[::-1]
        if mask.any(): result.append(dict(rle=encode(mask),score=float(instances.scores[idx]),view=view))
    del x, instances
    return result

def collect(directory, names, views, filename):
    path = OUT/filename
    cache = json.loads(path.read_text()) if path.exists() else {}
    started=time.monotonic()
    for i,name in enumerate(names,1):
        cache.setdefault(name,{})
        for view in views:
            if view not in cache[name]:
                cache[name][view] = candidates(directory/name,view)
                temp=path.with_suffix('.tmp');temp.write_text(json.dumps(cache));temp.replace(path)
        if i%10==0 or i==len(names):
            print('CANDIDATES',filename,i,'/',len(names),'elapsed_s',round(time.monotonic()-started),flush=True)
    return cache

def consensus(per_view):
    pool=[p for v in VIEWS for p in per_view[v]]
    if not pool: return []
    iou=mu.iou([p['rle'] for p in pool],[p['rle'] for p in pool],[0]*len(pool))
    order=sorted(range(len(pool)),key=lambda i:pool[i]['score'],reverse=True)
    used=set();result=[]
    for i in order:
        if i in used: continue
        matched=[i]
        for v in VIEWS:
            if v==pool[i]['view']: continue
            options=[j for j in order if j not in used and pool[j]['view']==v and iou[i,j]>.5]
            if options: matched.append(max(options,key=lambda j:iou[i,j]))
        used.update(matched)
        if len(matched)>=2:
            result.append(dict(rle=pool[i]['rle'],score=sum(pool[j]['score'] for j in matched)/len(matched),
                               support=len(matched)))
    return result

def disjoint(pool):
    occupied=np.zeros((2048,2048),bool); result=[]
    for p in sorted(pool,key=lambda p:p['score'],reverse=True):
        mask=mu.decode(p['rle']).astype(bool)
        mask &= ~occupied
        if not mask.any(): continue
        occupied |= mask
        rle=encode(mask)
        result.append(dict(rle=rle,score=p['score'],area=int(mu.area(rle))))
    return result

# Fixed-mask boundary experiment: no training; one forward reused for three contours.
from datetime import datetime, timezone
import math
import torch.nn.functional as F
from detectron2.structures import ImageList

CUTOFF = datetime.fromisoformat('2026-10-16T23:35:00+00:00').timestamp()
assert time.time() < CUTOFF, 'Current quota cycle cutoff reached'
BOUNDARY_LEVELS=(.35,.5,.65)
OUT=Path('/kaggle/working/boundary_results')
OUT.mkdir(exist_ok=True)
LIMIT_SECONDS=8*3600
source=Path('/kaggle/working/finetune_results/model_best_finetune.pth')
assert source.exists(), 'Previous best checkpoint must remain in the preserved working directory'
assert hashlib.sha256(source.read_bytes()).hexdigest()=='6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1'
state=torch.load(source,map_location='cpu',weights_only=True)
assert state['finetune_step']==300
model.load_state_dict(state['model'],strict=True)
del state
model.requires_grad_(False).eval()

@torch.inference_mode()
def boundary_candidates(path,view,levels):
    assert time.time()<CUTOFF, 'Current quota cycle cutoff reached'
    image=utils.read_image(str(path),format=cfg.INPUT.FORMAT)
    assert image.shape[:2]==(2048,2048)
    if view in ('hflip','rot180'):image=image[:,::-1]
    if view in ('vflip','rot180'):image=image[::-1]
    tensor=torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32).to(model.device)
    images=ImageList.from_tensors([(tensor-model.pixel_mean)/model.pixel_std],model.size_divisibility)
    predictions=model.sem_seg_head(model.backbone(images.tensor))
    cls=predictions['pred_logits'][0];masks=predictions['pred_masks'][0]
    scores,indices=cls.softmax(-1)[:,:-1].flatten().topk(model.test_topk_per_image,sorted=False)
    queries=indices//(cls.shape[-1]-1)
    valid=scores>=.15;scores,queries=scores[valid],queries[valid]
    result={str(p):[] for p in levels}
    for start in range(0,len(queries),4):
        logits=F.interpolate(masks[queries[start:start+4]][None],size=images.tensor.shape[-2:],mode='bilinear',align_corners=False)[0]
        logits=logits[:,:images.image_sizes[0][0],:images.image_sizes[0][1]]
        logits=F.interpolate(logits[None],size=(2048,2048),mode='bilinear',align_corners=False)[0]
        canonical=logits>0
        quality=(logits.sigmoid().flatten(1)*canonical.flatten(1)).sum(1)/(canonical.flatten(1).sum(1)+1e-6)
        final_scores=scores[start:start+4]*quality
        for level in levels:
            binary=(logits>math.log(level/(1-level))).cpu().numpy()
            for mask,score in zip(binary,final_scores.cpu().tolist()):
                if score<.15 or not mask.any():continue
                if view in ('hflip','rot180'):mask=mask[:,::-1]
                if view in ('vflip','rot180'):mask=mask[::-1]
                result[str(level)].append(dict(rle=encode(mask),score=score,view=view))
    for key in result:result[key].sort(key=lambda p:p['score'],reverse=True)
    return result

def collect_boundary(directory,names,levels,filename):
    path=OUT/filename
    cache=json.loads(path.read_text()) if path.exists() else {}
    for i,name in enumerate(names,1):
        cache.setdefault(name,{})
        for view in VIEWS:
            if view not in cache[name]:
                cache[name][view]=boundary_candidates(directory/name,view,levels)
                temporary=path.with_suffix('.tmp');temporary.write_text(json.dumps(cache));temporary.replace(path)
        if i%10==0 or i==len(names):print('BOUNDARY_CACHE',filename,i,'/',len(names),flush=True)
    return cache

def boundary_predictions(cache,level):
    return {name:[p['rle'] for p in disjoint(consensus({v:cache[name][v][str(level)] for v in VIEWS}))
                  if p['score']>=.9 and p['area']>=256] for name in cache}

def boundary_score(predictions,split):
    counts=[]
    for r in records:
        if part[r['file_name']]!=split:continue
        truth=[mask_rle(a['segmentation'],2048,2048) for a in gt[r['source_image_id']]]
        pred=predictions[r['file_name']]
        iou=mu.iou(truth,pred,[0]*len(pred)) if pred else np.zeros((len(truth),0))
        counts.append(pq_counts(iou))
    return summarize_counts(counts)

print('BOUNDARY_START',json.dumps(dict(levels=BOUNDARY_LEVELS,checkpoint=str(source),training=False,cutoff_utc='2026-10-16T23:35:00Z')),flush=True)
cal_names=sorted(n for n in part if part[n]=='calibration')
cal_cache=collect_boundary(train_dir,cal_names,BOUNDARY_LEVELS,'calibration_candidates.json')
grid=[dict(level=level,calibration=boundary_score(boundary_predictions(cal_cache,level),'calibration')) for level in BOUNDARY_LEVELS]
canonical=next(g for g in grid if g['level']==.5)
assert abs(canonical['calibration']['PQ']-.4563696594893772)<1e-7, canonical
chosen=max(grid,key=lambda x:x['calibration']['PQ'])
summary=dict(grid=grid,chosen=chosen,holdout_accepted=False,baseline_public_score=.36,
             baseline_calibration_pq=.4563696594893772,baseline_holdout_pq=.40760711778568315,
             trained=False,source_sha256=hashlib.sha256(source.read_bytes()).hexdigest())
(OUT/'boundary_summary.json').write_text(json.dumps(summary,indent=2))
print('BOUNDARY_CALIBRATION',json.dumps(summary),flush=True)
if chosen['level']!=.5 and chosen['calibration']['PQ']>.4563696594893772+.003:
    hold_names=sorted(n for n in part if part[n]=='holdout')
    hold_cache=collect_boundary(train_dir,hold_names,(chosen['level'],),'holdout_candidates.json')
    hold=boundary_score(boundary_predictions(hold_cache,chosen['level']),'holdout')
    summary['holdout']=hold
    summary['holdout_accepted']=hold['PQ']>.40760711778568315+.003
    (OUT/'boundary_summary.json').write_text(json.dumps(summary,indent=2))
    print('BOUNDARY_HOLDOUT',json.dumps(hold),flush=True)
    if summary['holdout_accepted']:
        test_names=sorted(p.name for p in test_dir.glob('*.jpeg'));assert len(test_names)==180
        test_cache=collect_boundary(test_dir,test_names,(chosen['level'],),'test_candidates.json')
        predictions=boundary_predictions(test_cache,chosen['level']);rows=[]
        for name in test_names:
            rows.extend((Path(name).stem+'_'+str(i),r['counts']) for i,r in enumerate(predictions[name],1))
        with (OUT/'submission_boundary.csv').open('w',newline='') as f:
            writer=csv.writer(f);writer.writerow(['filament_id','segmentation_rle']);writer.writerows(rows)
        summary.update(test_images=180,instances=len(rows))
summary.update(elapsed_seconds=round(time.monotonic()-RUN_STARTED),completed_at_utc=datetime.now(timezone.utc).isoformat())
(OUT/'boundary_summary.json').write_text(json.dumps(summary,indent=2))
with zipfile.ZipFile('/kaggle/working/boundary_results.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
    for p in OUT.iterdir():archive.write(p,p.name)
print('BOUNDARY_COMPLETE',json.dumps(summary),flush=True)


ASSETS: /kaggle/working/solar_assets
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu128 CUDA: 12.8
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 8.5 MB/s eta 0:00:00
Building Detectron2 CPU extension; log: /kaggle/working/detectron2_install.log
Processing ./solar_m2f_runtime/detectron2
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for detectron2: filename=detectron2-0.6-cp313-cp313-linux_x86_64.whl size=6675511 sha256=bc1241a22600ecfbe318b3cbc738165785e98d700efcc0bbf46f73a697a4592c
  Stored in directory: /tmp/pip-ephem-wheel-cache-kkgyi66n/wheels/8e/7

/kaggle/working/solar_assets/solar_runtime.py:48: UserWarning: Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.
  warnings.warn('Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.')
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/ops/modules/ms_deform_attn.py:87: SyntaxWarning: invalid escape sequence '\s'
  :param input_flatten               (N, \sum_{l=0}^{L-1} H_l \cdot W_l, C)
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/msdeformattn.py:314: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


Runtime ready: 0.6 pytorch_reference
SETUP_SECONDS: 100
BOUNDARY_START {"levels": [0.35, 0.5, 0.65], "checkpoint": "/kaggle/working/finetune_results/model_best_finetune.pth", "training": false, "cutoff_utc": "2026-10-16T23:35:00Z"}


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


BOUNDARY_CACHE calibration_candidates.json 10 / 70
BOUNDARY_CACHE calibration_candidates.json 20 / 70
BOUNDARY_CACHE calibration_candidates.json 30 / 70
BOUNDARY_CACHE calibration_candidates.json 40 / 70
BOUNDARY_CACHE calibration_candidates.json 50 / 70
BOUNDARY_CACHE calibration_candidates.json 60 / 70
BOUNDARY_CACHE calibration_candidates.json 70 / 70
BOUNDARY_CALIBRATION {"grid": [{"level": 0.35, "calibration": {"PQ": 0.45039650835719797, "SQ": 0.67877296370364, "RQ": 0.6635451505016723, "tp": 496, "fp": 217, "fn": 286, "iou_sum": 336.67138999700546}}, {"level": 0.5, "calibration": {"PQ": 0.4563696594893772, "SQ": 0.6799724645403973, "RQ": 0.6711590296495957, "tp": 498, "fp": 204, "fn": 284, "iou_sum": 338.62628734111786}}, {"level": 0.65, "calibration": {"PQ": 0.4588468780569024, "SQ": 0.6804699201583863, "RQ": 0.6743088334457181, "tp": 500, "fp": 201, "fn": 282, "iou_sum": 340.2349600791931}}], "chosen": {"level": 0.65, "calibration": {"PQ": 0.4588468780569024, "SQ": 0.6804699201

In [1]:
import os, sys, time, json, shutil, subprocess
from pathlib import Path

RUN_STARTED = time.monotonic()
import zipfile
archives = list(Path('/kaggle/input').rglob('solar_payload.bin'))
assert len(archives) == 1, archives
ASSETS = Path('/kaggle/working/solar_assets')
ASSETS.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    archive.extractall(ASSETS)
INPUTS = ASSETS
RUNTIME = Path('/kaggle/working/solar_m2f_runtime')
RUNTIME.mkdir(exist_ok=True)
for name in ['Mask2Former', 'detectron2']:
    target = RUNTIME / name
    if not target.exists():
        shutil.copytree(ASSETS / 'vendor' / name, target)
print('ASSETS:', ASSETS, flush=True)
print('Python:', sys.version, flush=True)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, flush=True)
assert torch.cuda.is_available(), 'A free Kaggle GPU is required; no CPU full run.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ninja', 'pycocotools', 'timm==0.9.16', 'fvcore==0.1.5.post20221221',
    'iopath==0.1.9', 'yacs', 'omegaconf>=2.1,<2.4', 'hydra-core',
    'termcolor', 'tabulate', 'cloudpickle', 'tensorboard', 'shapely',
    'scipy', 'submitit', 'black', 'packaging'], check=True, timeout=900)
# The Mask2Former R50 path needs no Detectron2 CUDA-specific operators.
# Build its normal CPU extension; deformable attention uses the existing
# project's differentiable PyTorch reference implementation during inference.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', MAX_JOBS='2')
log_path = Path('/kaggle/working/detectron2_install.log')
print('Building Detectron2 CPU extension; log:', log_path, flush=True)
with log_path.open('w') as log:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install',
        '--no-build-isolation', '--no-deps', str(RUNTIME / 'detectron2')],
        env=env, stdout=log, stderr=subprocess.STDOUT, timeout=1800)
print(log_path.read_text()[-8000:], flush=True)
assert result.returncode == 0, 'See detectron2_install.log'
sys.path[:0] = [str(ASSETS), str(RUNTIME / 'Mask2Former')]
from solar_runtime import install_msda_backend
BACKEND = install_msda_backend('torch')
import detectron2, mask2former
print('Runtime ready:', detectron2.__version__, BACKEND, flush=True)
print('SETUP_SECONDS:', round(time.monotonic() - RUN_STARTED), flush=True)


ASSETS: /kaggle/working/solar_assets
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu128 CUDA: 12.8
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.4 MB/s eta 0:00:00
Building Detectron2 CPU extension; log: /kaggle/working/detectron2_install.log
Processing ./solar_m2f_runtime/detectron2
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for detectron2: filename=detectron2-0.6-cp313-cp313-linux_x86_64.whl size=6675511 sha256=1d73f2ffd14ae46a4afc9444322036edf08b196ce37f394deb437732a1f183fd
  Stored in directory: /tmp/pip-ephem-wheel-cache-dmry6jmx/wheels/8e/7f

/kaggle/working/solar_assets/solar_runtime.py:48: UserWarning: Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.
  warnings.warn('Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.')
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/ops/modules/ms_deform_attn.py:87: SyntaxWarning: invalid escape sequence '\s'
  :param input_flatten               (N, \sum_{l=0}^{L-1} H_l \cdot W_l, C)
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/msdeformattn.py:314: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


Runtime ready: 0.6 pytorch_reference
SETUP_SECONDS: 112


In [2]:
import csv, gc, hashlib, itertools, json, time, zipfile
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from pycocotools import mask as mu
from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.modeling import build_model
from detectron2.data import detection_utils as utils
from mask2former import add_maskformer2_config
from solar_inference import install_chunked_instance_inference
from solar_evaluation import mask_rle, pq_counts, summarize_counts

OUT = Path('/kaggle/working/calibration_results')
OUT.mkdir(exist_ok=True)
LIMIT_SECONDS = 5.5 * 3600
torch.set_grad_enabled(False)
torch.manual_seed(42)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
provenance = json.loads((ASSETS / 'provenance.json').read_text())
weight = ASSETS / 'model_best.pth'
assert hashlib.sha256(weight.read_bytes()).hexdigest() == provenance['checkpoint_sha256']
cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
cfg.set_new_allowed(True)
cfg.merge_from_file(str(ASSETS / 'config.yaml'))
cfg.MODEL.DEVICE = 'cuda'
cfg.DATASETS.TRAIN = ('solar_calibration_metadata',)
cfg.DATASETS.TEST = ()
model = build_model(cfg)
checkpoint = torch.load(weight, map_location='cpu', weights_only=True)
assert checkpoint['iteration'] == 36999
model.load_state_dict(checkpoint['model'], strict=True)
del checkpoint
gc.collect()
model.requires_grad_(False).eval()
install_chunked_instance_inference(model, .15)
records = json.loads((ASSETS / 'validation_records.json').read_text())
dates = sorted({r['file_name'][:8] for r in records},
               key=lambda x: hashlib.sha256(('calibration-v1-' + x).encode()).hexdigest())
cal_dates = set(dates[:len(dates)//2])
part = {r['file_name']: ('calibration' if r['file_name'][:8] in cal_dates else 'holdout') for r in records}
assert {r['file_name'][:8] for r in records if part[r['file_name']]=='calibration'}.isdisjoint(
       {r['file_name'][:8] for r in records if part[r['file_name']]=='holdout'})
(OUT/'split.json').write_text(json.dumps(part, indent=2))
root = next(ASSETS.rglob('test_images')).parent.parent
train_dir, test_dir = root/'train/train_images', root/'test/test_images'
raw = json.loads((root/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
gt = defaultdict(list)
for a in raw['annotations']:
    gt[a['image_id']].append(a)
VIEWS = ('native','hflip','vflip','rot180')

def encode(mask):
    r = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    r['counts'] = r['counts'].decode('ascii')
    return r

def candidates(path, view):
    assert time.monotonic()-RUN_STARTED < LIMIT_SECONDS, 'Calibration time cap reached'
    assert not model.training and not torch.is_grad_enabled()
    image = utils.read_image(str(path), format=cfg.INPUT.FORMAT)
    assert image.shape[:2] == (2048,2048)
    if view in ('hflip','rot180'): image = image[:,::-1]
    if view in ('vflip','rot180'): image = image[::-1]
    x = torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
    with torch.inference_mode():
        instances = model([{'image':x,'height':2048,'width':2048}])[0]['instances']
    result = []
    for idx in instances.scores.argsort(descending=True).tolist():
        mask = instances.pred_masks[idx].numpy()
        if view in ('hflip','rot180'): mask = mask[:,::-1]
        if view in ('vflip','rot180'): mask = mask[::-1]
        if mask.any(): result.append(dict(rle=encode(mask),score=float(instances.scores[idx]),view=view))
    del x, instances
    return result

def collect(directory, names, views, filename):
    path = OUT/filename
    cache = json.loads(path.read_text()) if path.exists() else {}
    started=time.monotonic()
    for i,name in enumerate(names,1):
        cache.setdefault(name,{})
        for view in views:
            if view not in cache[name]:
                cache[name][view] = candidates(directory/name,view)
                temp=path.with_suffix('.tmp');temp.write_text(json.dumps(cache));temp.replace(path)
        if i%10==0 or i==len(names):
            print('CANDIDATES',filename,i,'/',len(names),'elapsed_s',round(time.monotonic()-started),flush=True)
    return cache

def consensus(per_view):
    pool=[p for v in VIEWS for p in per_view[v]]
    if not pool: return []
    iou=mu.iou([p['rle'] for p in pool],[p['rle'] for p in pool],[0]*len(pool))
    order=sorted(range(len(pool)),key=lambda i:pool[i]['score'],reverse=True)
    used=set();result=[]
    for i in order:
        if i in used: continue
        matched=[i]
        for v in VIEWS:
            if v==pool[i]['view']: continue
            options=[j for j in order if j not in used and pool[j]['view']==v and iou[i,j]>.5]
            if options: matched.append(max(options,key=lambda j:iou[i,j]))
        used.update(matched)
        if len(matched)>=2:
            result.append(dict(rle=pool[i]['rle'],score=sum(pool[j]['score'] for j in matched)/len(matched),
                               support=len(matched)))
    return result

def disjoint(pool):
    occupied=np.zeros((2048,2048),bool); result=[]
    for p in sorted(pool,key=lambda p:p['score'],reverse=True):
        mask=mu.decode(p['rle']).astype(bool)
        mask &= ~occupied
        if not mask.any(): continue
        occupied |= mask
        rle=encode(mask)
        result.append(dict(rle=rle,score=p['score'],area=int(mu.area(rle))))
    return result


# Run after the shared setup cell and calibration definitions (without its sweep).
import math, random
from detectron2.structures import Instances

TRAIN_SECONDS = 2 * 3600
MAX_STEPS = 3000
EVAL_EVERY = 300
CROP_SIZE = 1024
BASE_LR = 1e-5
OUT = Path('/kaggle/working/finetune_results')
OUT.mkdir(exist_ok=True)
archives=list(Path('/kaggle/input').rglob('training_images.bin'))
assert len(archives)==1, archives
training_root=Path('/kaggle/working/training_images')
training_root.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z: z.extractall(training_root)
manifest=json.loads((training_root/'training_manifest.json').read_text())
train_names=manifest['train_images']
assert len(train_names)==566 and not set(train_names)&set(part)
assert set(manifest['validation_images'])==set(part)
train_records=defaultdict(list)
for im in raw['images']:
    if im['file_name'] in set(train_names): train_records[im['file_name']].append(im['id'])
assert set(train_records)==set(train_names)
rng=random.Random(20261016)
model.requires_grad_(True)
model.backbone.requires_grad_(False)
parameters=[p for p in model.parameters() if p.requires_grad]
assert parameters
optimizer=torch.optim.AdamW([
    {'params':[p for p in parameters if p.ndim>1],'weight_decay':.05},
    {'params':[p for p in parameters if p.ndim<=1],'weight_decay':0.}],lr=BASE_LR)
scaler=torch.amp.GradScaler('cuda',init_scale=1024.)
PATIENCE=3
no_improvement=0

def sample_training():
    name=rng.choice(train_names)
    sid=rng.choice(train_records[name])
    image=utils.read_image(str(training_root/'train_images'/name),format=cfg.INPUT.FORMAT)
    y,x=rng.randrange(2048-CROP_SIZE+1),rng.randrange(2048-CROP_SIZE+1)
    image=image[y:y+CROP_SIZE,x:x+CROP_SIZE]
    masks=[]
    for a in gt[sid]:
        mask=mu.decode(mask_rle(a['segmentation'],2048,2048))[y:y+CROP_SIZE,x:x+CROP_SIZE]
        if mask.any(): masks.append(mask)
    masks=np.stack(masks) if masks else np.zeros((0,CROP_SIZE,CROP_SIZE),np.uint8)
    if rng.random()<.5: image=image[:,::-1];masks=masks[:,:,::-1]
    if rng.random()<.5: image=image[::-1];masks=masks[:,::-1,:]
    target=Instances((CROP_SIZE,CROP_SIZE))
    target.gt_classes=torch.zeros(len(masks),dtype=torch.int64)
    target.gt_masks=torch.from_numpy(np.ascontiguousarray(masks)).bool()
    return {'image':torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1))),
            'instances':target,'height':CROP_SIZE,'width':CROP_SIZE},name

def evaluate_checkpoint(split):
    model.eval()
    cache={};rows=[]
    with torch.inference_mode():
        for i,name in enumerate(sorted(n for n in part if part[n]==split),1):
            p=disjoint(candidates(train_dir/name,'native'))
            cache[name]=[x['rle'] for x in p if x['score']>=.5]
    for r in records:
        if part[r['file_name']]!=split:continue
        truth=[mask_rle(a['segmentation'],2048,2048) for a in gt[r['source_image_id']]]
        pred=cache[r['file_name']]
        iou=mu.iou(truth,pred,[0]*len(pred)) if pred else np.zeros((len(truth),0))
        rows.append(pq_counts(iou))
    return summarize_counts(rows)

def save_checkpoint(path,step,metric):
    state={'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
           'finetune_step':step,'source_iteration':36999,'calibration_metrics':metric,
           'source_checkpoint_sha256':provenance['checkpoint_sha256'],'crop_size':CROP_SIZE}
    if 'last' in path.stem:
        state.update(optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
                     random_state=rng.getstate(),torch_rng=torch.get_rng_state(),
                     cuda_rng=torch.cuda.get_rng_state_all(),
                     elapsed_train_seconds=train_elapsed_before+time.monotonic()-train_started)
    temporary=path.with_suffix('.tmp')
    torch.save(state,temporary)
    temporary.replace(path)

baseline_cal=evaluate_checkpoint('calibration')
assert abs(baseline_cal['PQ']-.3137662575115845)<.003,baseline_cal
best_metric=baseline_cal
best_step=0
best_path=OUT/'model_best_finetune.pth'
if best_path.exists():
    previous=torch.load(best_path,map_location='cpu',weights_only=True)
    assert previous['source_checkpoint_sha256']==provenance['checkpoint_sha256']
    assert previous['crop_size']==CROP_SIZE
    best_metric,best_step=previous['calibration_metrics'],previous['finetune_step']
    del previous
else:
    save_checkpoint(best_path,0,best_metric)
history=[]
train_elapsed_before=0.
starting_step=0
last_path=OUT/'model_last_finetune.pth'
if last_path.exists():
    previous=torch.load(last_path,map_location='cpu',weights_only=True)
    assert previous['source_checkpoint_sha256']==provenance['checkpoint_sha256']
    assert previous['crop_size']==CROP_SIZE
    model.load_state_dict(previous['model'],strict=True)
    optimizer.load_state_dict(previous['optimizer']);scaler.load_state_dict(previous['scaler'])
    rng.setstate(previous['random_state']);torch.set_rng_state(previous['torch_rng'])
    torch.cuda.set_rng_state_all(previous['cuda_rng'])
    train_elapsed_before=previous['elapsed_train_seconds'];starting_step=previous['finetune_step']
    del previous
    history_path=OUT/'training_history.json'
    if history_path.exists():history=json.loads(history_path.read_text())
    print('TRAIN_RESUME',starting_step,'elapsed_train_seconds',round(train_elapsed_before),flush=True)
train_started=time.monotonic()
training_steps=starting_step
stop_reason='max_steps'
print('TRAIN_START',json.dumps(dict(images=566,held_out_images=141,crop=CROP_SIZE,
       max_steps=MAX_STEPS,max_training_seconds=TRAIN_SECONDS,backbone_frozen=True,
       trainable_parameters=sum(p.numel() for p in parameters),baseline_calibration=baseline_cal)),flush=True)
for step in range(starting_step+1,MAX_STEPS+1):
    if train_elapsed_before+time.monotonic()-train_started>=TRAIN_SECONDS:
        stop_reason='training_time_cap';break
    model.train()
    record,name=sample_training()
    optimizer.zero_grad(set_to_none=True)
    with torch.enable_grad(), torch.autocast('cuda',dtype=torch.float16):
        losses=model([record])
        loss=sum(losses.values())
    assert torch.isfinite(loss), {k:float(v) for k,v in losses.items()}
    with torch.enable_grad():
        scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    norm=torch.nn.utils.clip_grad_norm_(parameters,.1)
    old_scale=scaler.get_scale()
    scaler.step(optimizer);scaler.update()
    updated=scaler.get_scale()>=old_scale
    if not updated:
        print('AMP_SKIPPED_UPDATE',step,'scale',scaler.get_scale(),flush=True)
    training_steps=step
    lr=BASE_LR*(.2+.8*(1+math.cos(math.pi*step/MAX_STEPS))/2)
    for group in optimizer.param_groups:group['lr']=lr
    history.append(dict(step=step,loss=float(loss),lr=lr,optimizer_updated=updated,
                        grad_norm=float(norm) if torch.isfinite(norm) else None))
    if step in (1,5,10) or step%50==0:
        print('TRAIN',step,'loss',round(float(loss),4),'elapsed_s',round(time.monotonic()-train_started),
              'peak_vram_gib',round(torch.cuda.max_memory_allocated()/1024**3,3),flush=True)
    del losses,loss,record
    if step%EVAL_EVERY==0 or step==MAX_STEPS:
        metric=evaluate_checkpoint('calibration')
        print('CHECKPOINT',step,json.dumps(metric),flush=True)
        history[-1]['calibration']=metric
        meaningful_gain=metric['PQ']>best_metric['PQ']+.0005
        no_improvement=0 if meaningful_gain else no_improvement+1
        if metric['PQ']>best_metric['PQ']:
            best_metric,best_step=metric,step
            save_checkpoint(OUT/'model_best_finetune.pth',step,metric)
        save_checkpoint(OUT/'model_last_finetune.pth',step,metric)
        (OUT/'training_history.json').write_text(json.dumps(history))
        if no_improvement>=PATIENCE:
            stop_reason='three_evaluations_without_meaningful_improvement'
            print('EARLY_STOP',step,stop_reason,flush=True)
            break
save_checkpoint(OUT/'model_last_finetune.pth',training_steps,None)
best=torch.load(OUT/'model_best_finetune.pth',map_location='cpu',weights_only=True)
model.load_state_dict(best['model'],strict=True);del best
def evaluate_fixed_consensus(split):
    selected=sorted(n for n in part if part[n]==split)
    cached=collect(train_dir,selected,VIEWS,'finetune_'+split+'_candidates.json')
    predictions={n:[x['rle'] for x in disjoint(consensus(cached[n]))
                    if x['score']>=.9 and x['area']>=256] for n in selected}
    rows=[]
    for r in records:
        if part[r['file_name']]!=split:continue
        truth=[mask_rle(a['segmentation'],2048,2048) for a in gt[r['source_image_id']]]
        pred=predictions[r['file_name']]
        iou=mu.iou(truth,pred,[0]*len(pred)) if pred else np.zeros((len(truth),0))
        rows.append(pq_counts(iou))
    return summarize_counts(rows)

# Weights use the original native-view proxy for selection; the final candidate
# must beat the current best pipeline with its already frozen postprocessor.
baseline_consensus_calibration_pq=.38339739457112637
baseline_consensus_holdout_pq=.34730657114948477
final_calibration=None;holdout=None;accept=False
if best_step>0 and best_metric['PQ']>baseline_cal['PQ']+.003:
    final_calibration=evaluate_fixed_consensus('calibration')
    if final_calibration['PQ']>baseline_consensus_calibration_pq+.003:
        holdout=evaluate_fixed_consensus('holdout')
        accept=holdout['PQ']>baseline_consensus_holdout_pq+.003
summary=dict(training_performed=True,training_steps=training_steps,best_step=best_step,
             baseline_calibration=baseline_cal,best_calibration=best_metric,holdout=holdout,
             baseline_consensus_calibration_pq=baseline_consensus_calibration_pq,
             baseline_consensus_holdout_pq=baseline_consensus_holdout_pq,
             fixed_consensus_calibration=final_calibration,holdout_accepted=bool(accept),
             stop_reason=stop_reason,early_stopping_patience=PATIENCE,
             current_best_public_score=.30,
             crop_size=CROP_SIZE,backbone_frozen=True,physical_image_uniform_sampling=True,
             original_checkpoint_sha256=provenance['original_checkpoint_sha256'],
             source_checkpoint_sha256=provenance['checkpoint_sha256'],
             peak_vram_gib=torch.cuda.max_memory_allocated()/1024**3,
             elapsed_seconds=round(time.monotonic()-RUN_STARTED),gpu=torch.cuda.get_device_name(0))
if accept:
    model.eval();rows=[]
    tests=sorted(p.name for p in test_dir.glob('*.jpeg'));assert len(tests)==180
    cached=collect(test_dir,tests,VIEWS,'finetune_test_candidates.json')
    for i,name in enumerate(tests,1):
        pred=[x['rle'] for x in disjoint(consensus(cached[name])) if x['score']>=.9 and x['area']>=256]
        rows.extend((Path(name).stem+'_'+str(k),x['counts']) for k,x in enumerate(pred,1))
        if i%10==0:print('FINETUNE_TEST',i,'/180',flush=True)
    with (OUT/'submission_finetune.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['filament_id','segmentation_rle']);w.writerows(rows)
    summary.update(processed_test_images=180,predicted_instances=len(rows))
(OUT/'training_history.json').write_text(json.dumps(history))
(OUT/'finetune_summary.json').write_text(json.dumps(summary,indent=2))
with zipfile.ZipFile('/kaggle/working/finetune_results.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():z.write(p,p.name)
print('FINETUNE_COMPLETE',json.dumps(summary),flush=True)


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


TRAIN_START {"images": 566, "held_out_images": 141, "crop": 1024, "max_steps": 3000, "max_training_seconds": 7200, "backbone_frozen": true, "trainable_parameters": 20495042, "baseline_calibration": {"PQ": 0.3137662575115845, "SQ": 0.6571119116143566, "RQ": 0.4774928774928775, "tp": 419, "fp": 554, "fn": 363, "iou_sum": 275.3298909664154}}


/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):


TRAIN 1 loss 44.0252 elapsed_s 2 peak_vram_gib 4.664


/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):


...

TRAIN 1150 loss 33.8286 elapsed_s 1498 peak_vram_gib 4.87


/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/kaggle/working/solar_m2f_runtime/Mask2Former/ma

TRAIN 1200 loss 13.3544 elapsed_s 1537 peak_vram_gib 4.87
CHECKPOINT 1200 {"PQ": 0.35644831568523083, "SQ": 0.6768046424166704, "RQ": 0.5266635205285513, "tp": 558, "fp": 779, "fn": 224, "iou_sum": 377.65699046850204}
EARLY_STOP 1200 three_evaluations_without_meaningful_improvement


Weight format of MaskFormerHead have changed! Please upgrade your models. Applying automatic conversion now ...


RuntimeError: Error(s) in loading state_dict for MaskFormer:
	Missing key(s) in state_dict: "sem_seg_head.pixel_decoder.input_proj.0.0.weight", "sem_seg_head.pixel_decoder.input_proj.0.0.bias", "sem_seg_head.pixel_decoder.input_proj.0.1.weight", "sem_seg_head.pixel_decoder.input_proj.0.1.bias", "sem_seg_head.pixel_decoder.input_proj.1.0.weight", "sem_seg_head.pixel_decoder.input_proj.1.0.bias", "sem_seg_head.pixel_decoder.input_proj.1.1.weight", "sem_seg_head.pixel_decoder.input_proj.1.1.bias", "sem_seg_head.pixel_decoder.input_proj.2.0.weight", "sem_seg_head.pixel_decoder.input_proj.2.0.bias", "sem_seg_head.pixel_decoder.input_proj.2.1.weight", "sem_seg_head.pixel_decoder.input_proj.2.1.bias", "sem_seg_head.pixel_decoder.transformer.level_embed", "sem_seg_head.pixel_decoder.transformer.encoder.layers.0.self_attn.sampling_offsets.weight", "sem_seg_head.pixel_decoder.transformer.encoder.layers.0.self_attn.sampling_offsets.bias", "sem_seg_head.pixel_decoder.transformer.encoder.layers.0.s

In [3]:
# Continue final evaluation from preserved training state; no optimizer updates.
import copy
from collections import OrderedDict
assert 'model' in globals() and 'OUT' in globals(), 'Keep the completed training kernel.'
best=torch.load(OUT/'model_best_finetune.pth',map_location='cpu',weights_only=True)
assert best['finetune_step']==300
assert best['source_checkpoint_sha256']==provenance['checkpoint_sha256']
assert set(best['model'])==set(model.state_dict())
restored=OrderedDict(best['model'])
restored._metadata=copy.deepcopy(model.state_dict()._metadata)
loaded=model.load_state_dict(restored,strict=True)
assert all(torch.equal(v.detach().cpu(),best['model'][k]) for k,v in model.state_dict().items())
print('RECOVERY_STRICT_LOAD_PASS',str(loaded),'step',best['finetune_step'],'all tensors identical',flush=True)
# Upgrade best/last disk files to preserve metadata for future independent loads.
for filename in ['model_best_finetune.pth','model_last_finetune.pth']:
    path=OUT/filename
    state=torch.load(path,map_location='cpu',weights_only=True)
    weights=OrderedDict(state['model'])
    weights._metadata=copy.deepcopy(model.state_dict()._metadata)
    state['model']=weights
    temporary=path.with_suffix('.tmp')
    torch.save(state,temporary);temporary.replace(path)
del best,restored,state,weights
model.requires_grad_(False).eval()
def evaluate_fixed_consensus(split):
    selected=sorted(n for n in part if part[n]==split)
    cached=collect(train_dir,selected,VIEWS,'finetune_'+split+'_candidates.json')
    predictions={n:[x['rle'] for x in disjoint(consensus(cached[n]))
                    if x['score']>=.9 and x['area']>=256] for n in selected}
    rows=[]
    for r in records:
        if part[r['file_name']]!=split:continue
        truth=[mask_rle(a['segmentation'],2048,2048) for a in gt[r['source_image_id']]]
        pred=predictions[r['file_name']]
        iou=mu.iou(truth,pred,[0]*len(pred)) if pred else np.zeros((len(truth),0))
        rows.append(pq_counts(iou))
    return summarize_counts(rows)

# Weights use the original native-view proxy for selection; the final candidate
# must beat the current best pipeline with its already frozen postprocessor.
baseline_consensus_calibration_pq=.38339739457112637
baseline_consensus_holdout_pq=.34730657114948477
final_calibration=None;holdout=None;accept=False
if best_step>0 and best_metric['PQ']>baseline_cal['PQ']+.003:
    final_calibration=evaluate_fixed_consensus('calibration')
    if final_calibration['PQ']>baseline_consensus_calibration_pq+.003:
        holdout=evaluate_fixed_consensus('holdout')
        accept=holdout['PQ']>baseline_consensus_holdout_pq+.003
summary=dict(training_performed=True,training_steps=training_steps,best_step=best_step,
             baseline_calibration=baseline_cal,best_calibration=best_metric,holdout=holdout,
             baseline_consensus_calibration_pq=baseline_consensus_calibration_pq,
             baseline_consensus_holdout_pq=baseline_consensus_holdout_pq,
             fixed_consensus_calibration=final_calibration,holdout_accepted=bool(accept),
             stop_reason=stop_reason,early_stopping_patience=PATIENCE,
             current_best_public_score=.30,
             crop_size=CROP_SIZE,backbone_frozen=True,physical_image_uniform_sampling=True,
             original_checkpoint_sha256=provenance['original_checkpoint_sha256'],
             source_checkpoint_sha256=provenance['checkpoint_sha256'],
             peak_vram_gib=torch.cuda.max_memory_allocated()/1024**3,
             elapsed_seconds=round(time.monotonic()-RUN_STARTED),gpu=torch.cuda.get_device_name(0))
if accept:
    model.eval();rows=[]
    tests=sorted(p.name for p in test_dir.glob('*.jpeg'));assert len(tests)==180
    cached=collect(test_dir,tests,VIEWS,'finetune_test_candidates.json')
    for i,name in enumerate(tests,1):
        pred=[x['rle'] for x in disjoint(consensus(cached[name])) if x['score']>=.9 and x['area']>=256]
        rows.extend((Path(name).stem+'_'+str(k),x['counts']) for k,x in enumerate(pred,1))
        if i%10==0:print('FINETUNE_TEST',i,'/180',flush=True)
    with (OUT/'submission_finetune.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['filament_id','segmentation_rle']);w.writerows(rows)
    summary.update(processed_test_images=180,predicted_instances=len(rows))
(OUT/'training_history.json').write_text(json.dumps(history))
(OUT/'finetune_summary.json').write_text(json.dumps(summary,indent=2))
with zipfile.ZipFile('/kaggle/working/finetune_results.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():z.write(p,p.name)
print('FINETUNE_COMPLETE',json.dumps(summary),flush=True)


RECOVERY_STRICT_LOAD_PASS <All keys matched successfully> step 300 all tensors identical
CANDIDATES finetune_calibration_candidates.json 10 / 70 elapsed_s 69
CANDIDATES finetune_calibration_candidates.json 20 / 70 elapsed_s 142
CANDIDATES finetune_calibration_candidates.json 30 / 70 elapsed_s 215
CANDIDATES finetune_calibration_candidates.json 40 / 70 elapsed_s 288
CANDIDATES finetune_calibration_candidates.json 50 / 70 elapsed_s 361
CANDIDATES finetune_calibration_candidates.json 60 / 70 elapsed_s 427
CANDIDATES finetune_calibration_candidates.json 70 / 70 elapsed_s 489
CANDIDATES finetune_holdout_candidates.json 10 / 71 elapsed_s 75
CANDIDATES finetune_holdout_candidates.json 20 / 71 elapsed_s 153
CANDIDATES finetune_holdout_candidates.json 30 / 71 elapsed_s 226
CANDIDATES finetune_holdout_candidates.json 40 / 71 elapsed_s 306
CANDIDATES finetune_holdout_candidates.json 50 / 71 elapsed_s 383
CANDIDATES finetune_holdout_candidates.json 60 / 71 elapsed_s 454
CANDIDATES finetune_holdout

In [ ]:
# Run only after completed boundary experiment definitions; never Run All.
from datetime import datetime, timezone
import warnings, copy, math, random
from collections import OrderedDict
from detectron2.structures import Instances
warnings.filterwarnings('ignore',category=FutureWarning)
CUTOFF=datetime.fromisoformat('2026-10-16T23:35:00+00:00').timestamp()
assert time.time()+3600<CUTOFF, 'Insufficient time in this free quota cycle'
assert not model.training
OUT=Path('/kaggle/working/context1536_results');OUT.mkdir(exist_ok=True)
SOURCE=Path('/kaggle/working/finetune_results/model_best_finetune.pth')
SOURCE_SHA='6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1'
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SOURCE_SHA
warm=torch.load(SOURCE,map_location='cpu',weights_only=True)
model.load_state_dict(warm['model'],strict=True);del warm
CROP_SIZE=1536;BASE_LR=2e-6;MAX_STEPS=1200;EVAL_EVERY=200;PATIENCE=2
TRAIN_SECONDS=min(7200,CUTOFF-time.time()-1200)
EXPERIMENT_STARTED=time.monotonic()
LIMIT_SECONDS=12*3600
archives=list(Path('/kaggle/input').rglob('training_images.bin'));assert len(archives)==1
training_root=Path('/kaggle/working/training_images');training_root.mkdir(exist_ok=True)
if not (training_root/'training_manifest.json').exists():
    with zipfile.ZipFile(archives[0]) as archive:archive.extractall(training_root)
manifest=json.loads((training_root/'training_manifest.json').read_text())
train_names=manifest['train_images'];assert len(train_names)==566 and not set(train_names)&set(part)
assert set(manifest['validation_images'])==set(part)
train_records=defaultdict(list)
for im in raw['images']:
    if im['file_name'] in set(train_names):train_records[im['file_name']].append(im['id'])
assert set(train_records)==set(train_names)
rng=random.Random(20261017)
model.requires_grad_(True);model.backbone.requires_grad_(False)
parameters=[p for p in model.parameters() if p.requires_grad]
optimizer=torch.optim.AdamW([{'params':[p for p in parameters if p.ndim>1],'weight_decay':.05},
                            {'params':[p for p in parameters if p.ndim<=1],'weight_decay':0.}],lr=BASE_LR)
scaler=torch.amp.GradScaler('cuda',init_scale=256.)

def sample_context():
    name=rng.choice(train_names);sid=rng.choice(train_records[name])
    image=utils.read_image(str(training_root/'train_images'/name),format=cfg.INPUT.FORMAT)
    y,x=rng.randrange(2048-CROP_SIZE+1),rng.randrange(2048-CROP_SIZE+1)
    image=image[y:y+CROP_SIZE,x:x+CROP_SIZE]
    masks=[]
    for a in gt[sid]:
        mask=mu.decode(mask_rle(a['segmentation'],2048,2048))[y:y+CROP_SIZE,x:x+CROP_SIZE]
        if mask.any():masks.append(mask)
    masks=np.stack(masks) if masks else np.zeros((0,CROP_SIZE,CROP_SIZE),np.uint8)
    if rng.random()<.5:image=image[:,::-1];masks=masks[:,:,::-1]
    if rng.random()<.5:image=image[::-1];masks=masks[:,::-1,:]
    target=Instances((CROP_SIZE,CROP_SIZE));target.gt_classes=torch.zeros(len(masks),dtype=torch.int64)
    target.gt_masks=torch.from_numpy(np.ascontiguousarray(masks)).bool()
    return {'image':torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1))),
            'instances':target,'height':CROP_SIZE,'width':CROP_SIZE},name

def load_context(weights):
    assert set(weights)==set(model.state_dict())
    restored=OrderedDict(weights);restored._metadata=copy.deepcopy(model.state_dict()._metadata)
    return model.load_state_dict(restored,strict=True)

def save_context(path,step,metric):
    state=model.state_dict();weights=OrderedDict((k,v.detach().cpu()) for k,v in state.items())
    weights._metadata=copy.deepcopy(state._metadata)
    output=dict(model=weights,context_step=step,crop_size=CROP_SIZE,source_sha256=SOURCE_SHA,
                calibration=metric,early_stopping_count=no_improvement,optimizer=optimizer.state_dict(),
                scaler=scaler.state_dict(),random_state=rng.getstate(),torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all(),best_step=best_step,best_metric=best_metric,
                elapsed_seconds=prior_elapsed+time.monotonic()-EXPERIMENT_STARTED)
    temp=path.with_suffix('.tmp');torch.save(output,temp);temp.replace(path)

def context_eval(split,step):
    assert time.time()<CUTOFF-60
    model.eval()
    names=sorted(n for n in part if part[n]==split)
    cache=collect(train_dir,names,VIEWS,f'{split}_step{step}_candidates.json')
    predictions={name:[p['rle'] for p in disjoint(consensus(cache[name])) if p['score']>=.9 and p['area']>=256] for name in names}
    counts=[]
    for record in records:
        if part[record['file_name']]!=split:continue
        truth=[mask_rle(a['segmentation'],2048,2048) for a in gt[record['source_image_id']]]
        prediction=predictions[record['file_name']]
        iou=mu.iou(truth,prediction,[0]*len(prediction)) if prediction else np.zeros((len(truth),0))
        counts.append(pq_counts(iou))
    return summarize_counts(counts)

no_improvement=0;best_step=0;best_metric=dict(PQ=.4563696594893772)
prior_elapsed=0.;starting_step=0;history=[]
last_path=OUT/'model_last_context.pth';best_path=OUT/'model_best_context.pth'
if last_path.exists():
    state=torch.load(last_path,map_location='cpu',weights_only=True)
    assert state['source_sha256']==SOURCE_SHA and state['crop_size']==CROP_SIZE
    load_context(state['model']);optimizer.load_state_dict(state['optimizer']);scaler.load_state_dict(state['scaler'])
    rng.setstate(state['random_state']);torch.set_rng_state(state['torch_rng']);torch.cuda.set_rng_state_all(state['cuda_rng'])
    starting_step=state['context_step'];no_improvement=state['early_stopping_count']
    best_step=state['best_step'];best_metric=state['best_metric'];prior_elapsed=state['elapsed_seconds']
    history=json.loads((OUT/'training_history.json').read_text()) if (OUT/'training_history.json').exists() else []
    del state
    print('CONTEXT_RESUME',starting_step,flush=True)
else:
    # Dry preflight computes gradients but performs no optimizer update.
    rng_state=rng.getstate();torch_state=torch.get_rng_state();cuda_state=torch.cuda.get_rng_state_all()
    model.train();optimizer.zero_grad(set_to_none=True);torch.cuda.reset_peak_memory_stats()
    record,name=sample_context()
    with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):
        losses=model([record]);loss=sum(losses.values())
    assert torch.isfinite(loss)
    with torch.enable_grad():scaler.scale(loss).backward()
    preflight=dict(crop=CROP_SIZE,optimizer_updates=0,loss=float(loss),peak_vram_gib=torch.cuda.max_memory_allocated()/1024**3)
    assert preflight['peak_vram_gib']<14.,preflight
    optimizer.zero_grad(set_to_none=True);del loss,losses,record
    rng.setstate(rng_state);torch.set_rng_state(torch_state);torch.cuda.set_rng_state_all(cuda_state)
    torch.cuda.empty_cache();model.eval()
    (OUT/'preflight.json').write_text(json.dumps(preflight,indent=2));print('CONTEXT_PREFLIGHT_PASS',json.dumps(preflight),flush=True)
    save_context(best_path,0,best_metric)
print('CONTEXT_TRAIN_START',json.dumps(dict(crop=CROP_SIZE,lr=BASE_LR,max_steps=MAX_STEPS,evaluate_every=EVAL_EVERY,
       baseline_calibration=best_metric,selection='fixed four-view .9/256 calibration PQ',wall_cap=TRAIN_SECONDS,
       trainable_parameters=sum(p.numel() for p in parameters))),flush=True)
stop_reason='max_steps';training_steps=starting_step
try:
    for step in range(starting_step+1,MAX_STEPS+1):
        if time.time()>CUTOFF-1200 or prior_elapsed+time.monotonic()-EXPERIMENT_STARTED>TRAIN_SECONDS:
            stop_reason='cycle_or_wall_cutoff';break
        model.train();optimizer.zero_grad(set_to_none=True);record,name=sample_context()
        with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):
            losses=model([record]);loss=sum(losses.values())
        assert torch.isfinite(loss)
        with torch.enable_grad():scaler.scale(loss).backward()
        scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(parameters,.1)
        old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()
        updated=scaler.get_scale()>=old_scale
        lr=BASE_LR*(.2+.8*(1+math.cos(math.pi*step/MAX_STEPS))/2)
        for group in optimizer.param_groups:group['lr']=lr
        training_steps=step
        history.append(dict(step=step,loss=float(loss),lr=lr,optimizer_updated=updated,
                            grad_norm=float(norm) if torch.isfinite(norm) else None))
        if step in (1,5,10) or step%50==0:
            print('CONTEXT_TRAIN',step,'loss',round(float(loss),4),'elapsed_s',round(time.monotonic()-EXPERIMENT_STARTED),
                  'peak_vram_gib',round(torch.cuda.max_memory_allocated()/1024**3,3),flush=True)
        del loss,losses,record
        if step%50==0:
            save_context(last_path,step,None);(OUT/'training_history.json').write_text(json.dumps(history))
        if step%EVAL_EVERY==0:
            metric=context_eval('calibration',step);history[-1]['calibration']=metric
            print('CONTEXT_CHECKPOINT',step,json.dumps(metric),flush=True)
            no_improvement=0 if metric['PQ']>best_metric['PQ']+.0005 else no_improvement+1
            if metric['PQ']>best_metric['PQ']:
                best_metric=metric;best_step=step;save_context(best_path,step,metric)
            save_context(last_path,step,metric);(OUT/'training_history.json').write_text(json.dumps(history))
            if no_improvement>=PATIENCE:stop_reason='early_stop';break
except BaseException:
    save_context(last_path,training_steps,None);(OUT/'training_history.json').write_text(json.dumps(history));raise
save_context(last_path,training_steps,None);(OUT/'training_history.json').write_text(json.dumps(history))
state=torch.load(best_path,map_location='cpu',weights_only=True);load_context(state['model']);del state
model.requires_grad_(False).eval()
summary=dict(crop=CROP_SIZE,lr=BASE_LR,training_steps=training_steps,best_step=best_step,best_calibration=best_metric,
             baseline_calibration_pq=.4563696594893772,baseline_holdout_pq=.40760711778568315,
             holdout_accepted=False,stop_reason=stop_reason,source_sha256=SOURCE_SHA,
             peak_vram_gib=torch.cuda.max_memory_allocated()/1024**3)
if best_step>0 and best_metric['PQ']>.4563696594893772+.003 and time.time()<CUTOFF-1500:
    holdout=context_eval('holdout',best_step);summary['holdout']=holdout
    summary['holdout_accepted']=holdout['PQ']>.40760711778568315+.003
    print('CONTEXT_HOLDOUT',json.dumps(holdout),flush=True)
    if summary['holdout_accepted'] and time.time()<CUTOFF-1500:
        names=sorted(p.name for p in test_dir.glob('*.jpeg'));assert len(names)==180
        cache=collect(test_dir,names,VIEWS,'test_candidates.json');rows=[]
        for name in names:
            prediction=[p['rle'] for p in disjoint(consensus(cache[name])) if p['score']>=.9 and p['area']>=256]
            rows.extend((Path(name).stem+'_'+str(i),p['counts']) for i,p in enumerate(prediction,1))
        with (OUT/'submission_context.csv').open('w',newline='') as f:
            writer=csv.writer(f);writer.writerow(['filament_id','segmentation_rle']);writer.writerows(rows)
        summary.update(test_images=180,instances=len(rows))
summary.update(elapsed_seconds=round(prior_elapsed+time.monotonic()-EXPERIMENT_STARTED),completed_at_utc=datetime.now(timezone.utc).isoformat())
(OUT/'context_summary.json').write_text(json.dumps(summary,indent=2))
with zipfile.ZipFile('/kaggle/working/context1536_results.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.iterdir():archive.write(path,path.name)
print('CONTEXT_COMPLETE',json.dumps(summary),flush=True)


CONTEXT_PREFLIGHT_PASS {"crop": 1536, "optimizer_updates": 0, "loss": 24.742225646972656, "peak_vram_gib": 10.21510934829712}
CONTEXT_TRAIN_START {"crop": 1536, "lr": 2e-06, "max_steps": 1200, "evaluate_every": 200, "baseline_calibration": {"PQ": 0.4563696594893772}, "selection": "fixed four-view .9/256 calibration PQ", "wall_cap": 7200, "trainable_parameters": 20495042}
CONTEXT_TRAIN 1 loss 24.7422 elapsed_s 5 peak_vram_gib 10.231
CONTEXT_TRAIN 5 loss 13.3175 elapsed_s 14 peak_vram_gib 10.61
CONTEXT_TRAIN 10 loss 16.1943 elapsed_s 24 peak_vram_gib 10.61
CONTEXT_TRAIN 50 loss 15.1545 elapsed_s 93 peak_vram_gib 10.61
CONTEXT_TRAIN 100 loss 16.7964 elapsed_s 180 peak_vram_gib 10.61
CONTEXT_TRAIN 150 loss 21.455 elapsed_s 270 peak_vram_gib 10.61
CONTEXT_TRAIN 200 loss 29.0027 elapsed_s 358 peak_vram_gib 10.61
CANDIDATES calibration_step200_candidates.json 10 / 70 elapsed_s 71
CANDIDATES calibration_step200_candidates.json 20 / 70 elapsed_s 148
CANDIDATES calibration_step200_candidates.jso

In [1]:
from pathlib import Path
import zipfile,json,time
root=Path('/kaggle/working/context1536_results')
assert (root/'context_summary.json').is_file()
print('BACKUP_START',json.loads((root/'context_summary.json').read_text()),flush=True)
with zipfile.ZipFile('/kaggle/working/context1536_results.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in root.iterdir():archive.write(path,path.name)
print('BACKUP_COMPLETE',Path('/kaggle/working/context1536_results.zip').stat().st_size,flush=True)


BACKUP_START {'crop': 1536, 'lr': 2e-06, 'training_steps': 600, 'best_step': 200, 'best_calibration': {'PQ': 0.46506024247668526, 'SQ': 0.6784520821316252, 'RQ': 0.685472496473907, 'tp': 486, 'fp': 150, 'fn': 296, 'iou_sum': 329.72771191596985}, 'baseline_calibration_pq': 0.4563696594893772, 'baseline_holdout_pq': 0.40760711778568315, 'holdout_accepted': False, 'stop_reason': 'early_stop', 'source_sha256': '6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1', 'peak_vram_gib': 10.609615325927734, 'holdout': {'PQ': 0.4087191251534633, 'SQ': 0.6717872462774578, 'RQ': 0.6084056037358239, 'tp': 456, 'fp': 200, 'fn': 387, 'iou_sum': 306.33498430252075}, 'elapsed_seconds': 3331, 'completed_at_utc': '2026-10-16T15:02:01.999429+00:00'}
BACKUP_COMPLETE 623042043


In [ ]:
import os, sys, time, json, shutil, subprocess
from pathlib import Path

RUN_STARTED = time.monotonic()
import zipfile
archives = list(Path('/kaggle/input').rglob('solar_payload.bin'))
assert len(archives) == 1, archives
ASSETS = Path('/kaggle/working/solar_assets')
ASSETS.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    archive.extractall(ASSETS)
INPUTS = ASSETS
RUNTIME = Path('/kaggle/working/solar_m2f_runtime')
RUNTIME.mkdir(exist_ok=True)
for name in ['Mask2Former', 'detectron2']:
    target = RUNTIME / name
    if not target.exists():
        shutil.copytree(ASSETS / 'vendor' / name, target)
print('ASSETS:', ASSETS, flush=True)
print('Python:', sys.version, flush=True)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, flush=True)
assert torch.cuda.is_available(), 'A free Kaggle GPU is required; no CPU full run.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ninja', 'pycocotools', 'timm==0.9.16', 'fvcore==0.1.5.post20221221',
    'iopath==0.1.9', 'yacs', 'omegaconf>=2.1,<2.4', 'hydra-core',
    'termcolor', 'tabulate', 'cloudpickle', 'tensorboard', 'shapely',
    'scipy', 'submitit', 'black', 'packaging'], check=True, timeout=900)
# The Mask2Former R50 path needs no Detectron2 CUDA-specific operators.
# Build its normal CPU extension; deformable attention uses the existing
# project's differentiable PyTorch reference implementation during inference.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', MAX_JOBS='2')
log_path = Path('/kaggle/working/detectron2_install.log')
print('Building Detectron2 CPU extension; log:', log_path, flush=True)
with log_path.open('w') as log:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install',
        '--no-build-isolation', '--no-deps', str(RUNTIME / 'detectron2')],
        env=env, stdout=log, stderr=subprocess.STDOUT, timeout=1800)
print(log_path.read_text()[-8000:], flush=True)
assert result.returncode == 0, 'See detectron2_install.log'
sys.path[:0] = [str(ASSETS), str(RUNTIME / 'Mask2Former')]
from solar_runtime import install_msda_backend
BACKEND = install_msda_backend('torch')
import detectron2, mask2former
print('Runtime ready:', detectron2.__version__, BACKEND, flush=True)
print('SETUP_SECONDS:', round(time.monotonic() - RUN_STARTED), flush=True)

import csv, gc, hashlib, itertools, json, time, zipfile
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from pycocotools import mask as mu
from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.modeling import build_model
from detectron2.data import detection_utils as utils
from mask2former import add_maskformer2_config
from solar_inference import install_chunked_instance_inference
from solar_evaluation import mask_rle, pq_counts, summarize_counts

OUT = Path('/kaggle/working/snake_pilot_20261017')
OUT.mkdir(exist_ok=True)
LIMIT_SECONDS = 5.5 * 3600
torch.set_grad_enabled(False)
torch.manual_seed(42)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
provenance = json.loads((ASSETS / 'provenance.json').read_text())
weight = ASSETS / 'model_best.pth'
assert hashlib.sha256(weight.read_bytes()).hexdigest() == provenance['checkpoint_sha256']
cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
cfg.set_new_allowed(True)
cfg.merge_from_file(str(ASSETS / 'config.yaml'))
cfg.MODEL.DEVICE = 'cuda'
cfg.DATASETS.TRAIN = ('solar_calibration_metadata',)
cfg.DATASETS.TEST = ()
model = build_model(cfg)
checkpoint = torch.load(weight, map_location='cpu', weights_only=True)
assert checkpoint['iteration'] == 36999
model.load_state_dict(checkpoint['model'], strict=True)
del checkpoint
gc.collect()
model.requires_grad_(False).eval()
install_chunked_instance_inference(model, .15)
records = json.loads((ASSETS / 'validation_records.json').read_text())
dates = sorted({r['file_name'][:8] for r in records},
               key=lambda x: hashlib.sha256(('calibration-v1-' + x).encode()).hexdigest())
cal_dates = set(dates[:len(dates)//2])
part = {r['file_name']: ('calibration' if r['file_name'][:8] in cal_dates else 'holdout') for r in records}
assert {r['file_name'][:8] for r in records if part[r['file_name']]=='calibration'}.isdisjoint(
       {r['file_name'][:8] for r in records if part[r['file_name']]=='holdout'})
(OUT/'split.json').write_text(json.dumps(part, indent=2))
root = next(ASSETS.rglob('test_images')).parent.parent
train_dir, test_dir = root/'train/train_images', root/'test/test_images'
raw = json.loads((root/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
gt = defaultdict(list)
for a in raw['annotations']:
    gt[a['image_id']].append(a)
VIEWS = ('native','hflip','vflip','rot180')

def encode(mask):
    r = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    r['counts'] = r['counts'].decode('ascii')
    return r

def candidates(path, view):
    assert time.monotonic()-RUN_STARTED < LIMIT_SECONDS, 'Calibration time cap reached'
    assert not model.training and not torch.is_grad_enabled()
    image = utils.read_image(str(path), format=cfg.INPUT.FORMAT)
    assert image.shape[:2] == (2048,2048)
    if view in ('hflip','rot180'): image = image[:,::-1]
    if view in ('vflip','rot180'): image = image[::-1]
    x = torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
    with torch.inference_mode():
        instances = model([{'image':x,'height':2048,'width':2048}])[0]['instances']
    result = []
    for idx in instances.scores.argsort(descending=True).tolist():
        mask = instances.pred_masks[idx].numpy()
        if view in ('hflip','rot180'): mask = mask[:,::-1]
        if view in ('vflip','rot180'): mask = mask[::-1]
        if mask.any(): result.append(dict(rle=encode(mask),score=float(instances.scores[idx]),view=view))
    del x, instances
    return result

def collect(directory, names, views, filename):
    path = OUT/filename
    cache = json.loads(path.read_text()) if path.exists() else {}
    started=time.monotonic()
    for i,name in enumerate(names,1):
        cache.setdefault(name,{})
        for view in views:
            if view not in cache[name]:
                cache[name][view] = candidates(directory/name,view)
                temp=path.with_suffix('.tmp');temp.write_text(json.dumps(cache));temp.replace(path)
        if i%10==0 or i==len(names):
            print('CANDIDATES',filename,i,'/',len(names),'elapsed_s',round(time.monotonic()-started),flush=True)
    return cache

def consensus(per_view):
    pool=[p for v in VIEWS for p in per_view[v]]
    if not pool: return []
    iou=mu.iou([p['rle'] for p in pool],[p['rle'] for p in pool],[0]*len(pool))
    order=sorted(range(len(pool)),key=lambda i:pool[i]['score'],reverse=True)
    used=set();result=[]
    for i in order:
        if i in used: continue
        matched=[i]
        for v in VIEWS:
            if v==pool[i]['view']: continue
            options=[j for j in order if j not in used and pool[j]['view']==v and iou[i,j]>.5]
            if options: matched.append(max(options,key=lambda j:iou[i,j]))
        used.update(matched)
        if len(matched)>=2:
            result.append(dict(rle=pool[i]['rle'],score=sum(pool[j]['score'] for j in matched)/len(matched),
                               support=len(matched)))
    return result

def disjoint(pool):
    occupied=np.zeros((2048,2048),bool); result=[]
    for p in sorted(pool,key=lambda p:p['score'],reverse=True):
        mask=mu.decode(p['rle']).astype(bool)
        mask &= ~occupied
        if not mask.any(): continue
        occupied |= mask
        rle=encode(mask)
        result.append(dict(rle=rle,score=p['score'],area=int(mu.area(rle))))
    return result



"""Bottleneck residual adapters inspired by Qi et al., DSConv (ICCV 2023).

Reference: https://arxiv.org/abs/2307.08388 (Sec. 3.1).
This is an independent 2D implementation, not a reproduction of DSCNet:
we use learned dense kernel weights, center-out cumulative transverse offsets,
bilinear sampling, and a zero-initialized residual gate around M2F features.
No DSCNet fusion strategy, topology loss, or medical data is included.
"""
import math

import torch
from torch import nn
from torch.nn import functional as F


class SnakeKernel2D(nn.Module):
    """Two connected K-point sampling paths, initialized as horizontal/vertical."""

    def __init__(self, channels, kernel_size=9, max_step=1., dynamic=True):
        super().__init__()
        if kernel_size < 3 or kernel_size % 2 != 1:
            raise ValueError('kernel_size must be odd and >= 3')
        self.channels = channels
        self.kernel_size = kernel_size
        self.max_step = float(max_step)
        self.dynamic = dynamic
        self.offset = nn.Conv2d(channels, 2 * (kernel_size - 1), 3, padding=1)
        nn.init.zeros_(self.offset.weight)
        nn.init.zeros_(self.offset.bias)
        self.weight = nn.Parameter(torch.empty(2, channels, channels, kernel_size))
        nn.init.normal_(self.weight, std=1. / math.sqrt(channels * kernel_size))
        if not dynamic:
            self.offset.requires_grad_(False)

    def transverse_offsets(self, x):
        b, _, h, w = x.shape
        k, radius = self.kernel_size, self.kernel_size // 2
        if not self.dynamic:
            return x.new_zeros((b, 2, k, h, w))
        steps = self.max_step * torch.tanh(self.offset(x))
        steps = steps.view(b, 2, k - 1, h, w)
        # On either side, steps are stored from the center towards the edge.
        left = steps[:, :, :radius].cumsum(2).flip(2)
        right = steps[:, :, radius:].cumsum(2)
        center = steps[:, :, :1] * 0.
        return torch.cat((left, center, right), dim=2)

    def forward(self, x):
        # Coordinates/sampling use FP32 for CUDA AMP compatibility and stability.
        # The inner 1x1 contractions may still be autocast by the caller.
        x = x.float()
        b, _, h, w = x.shape
        if min(h, w) <= 1:
            raise ValueError('Snake sampling requires H,W > 1')
        offsets = self.transverse_offsets(x).float()
        yy, xx = torch.meshgrid(torch.arange(h, device=x.device, dtype=x.dtype),
                                torch.arange(w, device=x.device, dtype=x.dtype), indexing='ij')
        output = []
        radius = self.kernel_size // 2
        for axis in range(2):
            value = None
            for idx in range(self.kernel_size):
                fixed = idx - radius
                if axis == 0:
                    gx, gy = xx + fixed, yy + offsets[:, axis, idx]
                else:
                    gx, gy = xx + offsets[:, axis, idx], yy + fixed
                gx = gx.expand(b, h, w)
                gy = gy.expand(b, h, w)
                grid = torch.stack((2 * gx / (w - 1) - 1, 2 * gy / (h - 1) - 1), dim=-1)
                # Sequential samples avoid a materialized B*C*K*H*W tensor.
                sampled = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True)
                weighted = F.conv2d(sampled, self.weight[axis, :, :, idx, None, None])
                value = weighted if value is None else value + weighted
            output.append(value)
        return torch.cat(output, dim=1)


class ResidualFeatureAdapter(nn.Module):
    def __init__(self, channels=256, bottleneck=32, mode='snake', kernel_size=9):
        super().__init__()
        if mode not in ('snake', 'fixed_snake', 'conv'):
            raise ValueError(mode)
        self.mode = mode
        self.reduce = nn.Sequential(nn.Conv2d(channels, bottleneck, 1),
                                    nn.GroupNorm(math.gcd(8, bottleneck), bottleneck), nn.GELU())
        self.spatial = (nn.Conv2d(bottleneck, 2 * bottleneck, 3, padding=1)
                        if mode == 'conv' else SnakeKernel2D(bottleneck, kernel_size,
                                                             dynamic=mode == 'snake'))
        self.expand = nn.Conv2d(2 * bottleneck, channels, 1)
        self.alpha = nn.Parameter(torch.zeros(()))

    def forward(self, x):
        correction = self.expand(F.gelu(self.spatial(self.reduce(x))))
        return x + self.alpha * correction.to(x.dtype)


def attach_feature_adapters(model, *, mode='snake', mask_features=True, scales=(), bottleneck=32, kernel_size=9):
    """Attach after loading original weights; expanded checkpoints attach before load.

    One mask adapter is shared by all query-decoder prediction heads; scale 0/1/2
    address the existing coarse-to-fine 1/32,1/16,1/8 features. The hook changes
    predictor inputs only. It never convolves query tokens or class logits.
    """
    head = model.sem_seg_head
    if hasattr(head, '_snake_feature_handle'):
        raise ValueError('Adapters already installed; remove before reinstalling')
    selected = tuple(sorted(set(scales)))
    if any(level not in (0, 1, 2) for level in selected):
        raise ValueError('scales must be a subset of (0,1,2)')
    if not mask_features and not selected:
        raise ValueError('Select at least one feature site')
    device = next(model.parameters()).device
    head.snake_mask_adapter = (ResidualFeatureAdapter(mode=mode, bottleneck=bottleneck, kernel_size=kernel_size)
                               if mask_features else nn.Identity()).to(device)
    head.snake_scale_adapters = nn.ModuleDict({str(i): ResidualFeatureAdapter(mode=mode, bottleneck=bottleneck,
                                                                           kernel_size=kernel_size)
                                              for i in selected}).to(device)
    config = dict(mode=mode, mask_features=mask_features, scales=list(selected),
                  channels=256, bottleneck=bottleneck, kernel_size=kernel_size,
                  reference='https://arxiv.org/abs/2307.08388', implementation='independent_bottleneck_residual_v1')
    head._snake_adapter_config = config

    def adapt_inputs(_module, args):
        features, masks, *rest = args
        if len(features) != 3:
            raise ValueError('Expected the original three M2F spatial feature levels')
        enhanced = [head.snake_scale_adapters[str(i)](feature) if str(i) in head.snake_scale_adapters else feature
                    for i, feature in enumerate(features)]
        return (enhanced, head.snake_mask_adapter(masks), *rest)

    head._snake_feature_handle = head.predictor.register_forward_pre_hook(adapt_inputs)
    return config


import ast,copy,random,warnings,traceback
from collections import OrderedDict
from detectron2.structures import Instances
warnings.filterwarnings('ignore',category=FutureWarning)
PILOT_STARTED=time.monotonic()
PILOT_WALL_CAP=5400
SOURCE=Path('/kaggle/working/finetune_results/model_best_finetune.pth')
SOURCE_SHA='6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1'
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SOURCE_SHA
base=torch.load(SOURCE,map_location='cpu',weights_only=True)
model.load_state_dict(base['model'],strict=True)
base_weights=base['model']
del base
training_root=Path('/kaggle/working/training_images')
if not (training_root/'training_manifest.json').exists():
    archives=list(Path('/kaggle/input').rglob('training_images.bin'));assert len(archives)==1
    training_root.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:z.extractall(training_root)
manifest=json.loads((training_root/'training_manifest.json').read_text())
train_names=manifest['train_images'];assert len(train_names)==566 and not set(train_names)&set(part)
train_records=defaultdict(list)
for image in raw['images']:
    if image['file_name'] in set(train_names):train_records[image['file_name']].append(image['id'])
CROP_SIZE=1024
def sample_training():
    name = rng.choice(train_names)
    sid = rng.choice(train_records[name])
    image = utils.read_image(str(training_root / 'train_images' / name), format=cfg.INPUT.FORMAT)
    y, x = (rng.randrange(2048 - CROP_SIZE + 1), rng.randrange(2048 - CROP_SIZE + 1))
    image = image[y:y + CROP_SIZE, x:x + CROP_SIZE]
    masks = []
    for a in gt[sid]:
        mask = mu.decode(mask_rle(a['segmentation'], 2048, 2048))[y:y + CROP_SIZE, x:x + CROP_SIZE]
        if mask.any():
            masks.append(mask)
    masks = np.stack(masks) if masks else np.zeros((0, CROP_SIZE, CROP_SIZE), np.uint8)
    if rng.random() < 0.5:
        image = image[:, ::-1]
        masks = masks[:, :, ::-1]
    if rng.random() < 0.5:
        image = image[::-1]
        masks = masks[:, ::-1, :]
    target = Instances((CROP_SIZE, CROP_SIZE))
    target.gt_classes = torch.zeros(len(masks), dtype=torch.int64)
    target.gt_masks = torch.from_numpy(np.ascontiguousarray(masks)).bool()
    return ({'image': torch.as_tensor(np.ascontiguousarray(image.transpose(2, 0, 1))), 'instances': target, 'height': CROP_SIZE, 'width': CROP_SIZE}, name)

def detach_adapters():
    h=model.sem_seg_head
    if hasattr(h,'_snake_feature_handle'):
        h._snake_feature_handle.remove()
        del h._snake_feature_handle
        del h.snake_mask_adapter,h.snake_scale_adapters
        del h._snake_adapter_config
    model.zero_grad(set_to_none=True)
    gc.collect();torch.cuda.empty_cache()

def create_arm(mode,scales=()):
    detach_adapters()
    model.load_state_dict(base_weights,strict=True)
    model.requires_grad_(False)
    torch.manual_seed(20261017)
    config=attach_feature_adapters(model,mode=mode,mask_features=True,scales=scales,bottleneck=32,kernel_size=9)
    return config

def checkpoint(path,config,step,optimizer,scaler,rng,history,metrics=None):
    adapter_state=OrderedDict((k,v.detach().cpu()) for k,v in model.state_dict().items() if '.snake_' in k)
    adapter_state._metadata=copy.deepcopy(model.state_dict()._metadata)
    data=dict(adapters=adapter_state,adapter_config=config,step=step,optimizer=optimizer.state_dict(),
              scaler=scaler.state_dict(),python_rng=rng.getstate(),numpy_rng=np.random.get_state(),
              torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all(),
              source_checkpoint_sha256=SOURCE_SHA,source_checkpoint=str(SOURCE),history=history,metrics=metrics,
              format='adapter_delta_v1; load original base first, attach adapters, then load adapter delta',
              quota_wall_cap_seconds=PILOT_WALL_CAP)
    tmp=path.with_suffix('.tmp');torch.save(data,tmp);tmp.replace(path)

def evaluate_arm(label):
    model.eval();cache={}
    selected=sorted(n for n in part if part[n]=='calibration')
    with torch.inference_mode():
        for idx,name in enumerate(selected,1):
            assert time.monotonic()-PILOT_STARTED<PILOT_WALL_CAP
            cache[name]={view:candidates(train_dir/name,view) for view in VIEWS}
            if idx%10==0:print('SNAKE_EVAL_PROGRESS',label,idx,'/70',flush=True)
    (OUT/(label+'_candidates.json')).write_text(json.dumps(cache))
    preds={name:[p['rle'] for p in disjoint(consensus(views)) if p['score']>=.9 and p['area']>=256]
           for name,views in cache.items()}
    rows=[]
    for record in records:
        name=record['file_name']
        if name not in preds:continue
        truth=[mask_rle(a['segmentation'],2048,2048) for a in gt[record['source_image_id']]]
        pred=preds[name]
        mat=mu.iou(truth,pred,[0]*len(pred)) if pred else np.zeros((len(truth),0))
        rows.append(pq_counts(mat))
    return summarize_counts(rows)

summary=dict(source_sha256=SOURCE_SHA,baseline_calibration_pq=.4563696594893772,
             crop=1024,steps_per_arm=200,evaluation='70 calibration physical images only; locked2048FP32/4view/.90/256',
             preflights=[],arms=[],training='base frozen; newly added residual adapters only',
             submitted=False,review_evaluated=False)
print('SNAKE_PILOT_START',json.dumps(summary),flush=True)
try:
    # Verify attachment preserves the actual pretrained full2048 inference.
    name=sorted(n for n in part if part[n]=='calibration')[0]
    model.eval()
    with torch.inference_mode():baseline_candidates=candidates(train_dir/name,'native')
    create_arm('snake',(0,1,2));model.eval()
    with torch.inference_mode():identity_candidates=candidates(train_dir/name,'native')
    assert baseline_candidates==identity_candidates,'Zero-gate identity regression'
    summary['actual_2048_zero_gate_identity']=True
    # Probe all spatial sites with an active gate and true backwards, no updates.
    model.train();model.backbone.eval();rng=random.Random(20261017)
    for adapter in [model.sem_seg_head.snake_mask_adapter,*model.sem_seg_head.snake_scale_adapters.values()]:
        adapter.alpha.data.fill_(.01)
    sample,_=sample_training();torch.cuda.reset_peak_memory_stats()
    with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):
        losses=model([sample]);loss=sum(losses.values())
    with torch.enable_grad():loss.backward()
    assert torch.isfinite(loss) and all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
    probe=dict(mode='snake_mask_plus_all_three_scales',loss=float(loss.detach()),
               peak_vram_gib=torch.cuda.max_memory_allocated()/1024**3,optimizer_updates=0)
    summary['preflights'].append(probe);print('SNAKE_ALL_SCALE_PREFLIGHT_PASS',json.dumps(probe),flush=True)
    del losses,loss,sample
    detach_adapters()
    for mode in ('conv','snake'):
        assert time.monotonic()-PILOT_STARTED+1200<PILOT_WALL_CAP,'Not enough time for another pilot arm'
        config=create_arm(mode);model.train();model.backbone.eval()
        params=[p for p in model.parameters() if p.requires_grad]
        gate=[p for n,p in model.named_parameters() if p.requires_grad and n.endswith('.alpha')]
        others=[p for n,p in model.named_parameters() if p.requires_grad and not n.endswith('.alpha')]
        optimizer=torch.optim.AdamW([dict(params=others,lr=1e-4),dict(params=gate,lr=1e-3)],weight_decay=.01)
        scaler=torch.amp.GradScaler('cuda',init_scale=64.)
        rng=random.Random(20261018);torch.manual_seed(20261018)
        history=[];updates=0;arm_started=time.monotonic();torch.cuda.reset_peak_memory_stats()
        for step in range(1,201):
            assert time.monotonic()-PILOT_STARTED<PILOT_WALL_CAP-900
            item,filename=sample_training();optimizer.zero_grad(set_to_none=True)
            with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):
                losses=model([item]);loss=sum(losses.values())
            assert torch.isfinite(loss),'Nonfinite loss'
            previous_scale=scaler.get_scale()
            with torch.enable_grad():scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            norm=torch.nn.utils.clip_grad_norm_(params,1.)
            scaler.step(optimizer);scaler.update()
            updates+=int(scaler.get_scale()>=previous_scale)
            history.append(dict(step=step,loss=float(loss.detach()),file_name=filename,
                                gradient_norm=float(norm),alpha=float(gate[0].detach()),scale=scaler.get_scale()))
            del losses,loss,item
            if step==1 or step%25==0:
                print('SNAKE_TRAIN',mode,step,json.dumps(history[-1]),'elapsed_s',round(time.monotonic()-arm_started),flush=True)
            if step%50==0:checkpoint(OUT/(mode+'_last.pth'),config,step,optimizer,scaler,rng,history)
        assert all(torch.equal(v.detach().cpu(),base_weights[k]) for k,v in model.state_dict().items() if '.snake_' not in k),'Frozen base changed'
        train_peak=torch.cuda.max_memory_allocated()/1024**3
        model.zero_grad(set_to_none=True)
        metrics=evaluate_arm(mode)
        checkpoint(OUT/(mode+'_last.pth'),config,200,optimizer,scaler,rng,history,metrics)
        result=dict(mode=mode,config=config,training_steps=200,optimizer_updates=updates,
                    trainable_parameters=sum(p.numel() for p in params),training_peak_vram_gib=train_peak,
                    calibration=metrics,elapsed_seconds=round(time.monotonic()-arm_started))
        summary['arms'].append(result)
        (OUT/(mode+'_history.json')).write_text(json.dumps(history))
        (OUT/'pilot_summary.json').write_text(json.dumps(summary,indent=2))
        print('SNAKE_ARM_COMPLETE',json.dumps(result),flush=True)
        del optimizer,scaler,params,gate,others
        detach_adapters()
    summary['status']='complete'
except Exception as error:
    summary['status']='error';summary['error_type']=type(error).__name__;summary['error']=str(error)
    traceback.print_exc()
finally:
    summary['elapsed_seconds']=round(time.monotonic()-PILOT_STARTED)
    (OUT/'pilot_summary.json').write_text(json.dumps(summary,indent=2))
    with zipfile.ZipFile('/kaggle/working/snake_pilot_20261017.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
        for path in OUT.iterdir():archive.write(path,path.name)
    model.eval();model.zero_grad(set_to_none=True);torch.cuda.empty_cache()
    print('SNAKE_PILOT_COMPLETE',json.dumps(summary),flush=True)


ASSETS: /kaggle/working/solar_assets
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu128 CUDA: 12.8
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 8.1 MB/s eta 0:00:00
Building Detectron2 CPU extension; log: /kaggle/working/detectron2_install.log
Processing ./solar_m2f_runtime/detectron2
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for detectron2: filename=detectron2-0.6-cp313-cp313-linux_x86_64.whl size=6675511 sha256=1cfd49f3878ea16e86fe31254fe82aa80c2485598cb1ddf3f4eed8584022541a
  Stored in directory: /tmp/pip-ephem-wheel-cache-9j2lqzi8/wheels/8e/7f

/kaggle/working/solar_assets/solar_runtime.py:48: UserWarning: Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.
  warnings.warn('Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.')
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/ops/modules/ms_deform_attn.py:87: SyntaxWarning: invalid escape sequence '\s'
  :param input_flatten               (N, \sum_{l=0}^{L-1} H_l \cdot W_l, C)
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/msdeformattn.py:314: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


Runtime ready: 0.6 pytorch_reference
SETUP_SECONDS: 118
SNAKE_PILOT_START {"source_sha256": "6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1", "baseline_calibration_pq": 0.4563696594893772, "crop": 1024, "steps_per_arm": 200, "evaluation": "70 calibration physical images only; locked2048FP32/4view/.90/256", "preflights": [], "arms": [], "training": "base frozen; newly added residual adapters only", "submitted": false, "review_evaluated": false}


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


SNAKE_ALL_SCALE_PREFLIGHT_PASS {"mode": "snake_mask_plus_all_three_scales", "loss": 35.411102294921875, "peak_vram_gib": 1.5825395584106445, "optimizer_updates": 0}
SNAKE_TRAIN conv 1 {"step": 1, "loss": 18.176288604736328, "file_name": "20131109190514Mh.jpeg", "gradient_norm": 1.2856251001358032, "alpha": -0.0010000000474974513, "scale": 64.0} elapsed_s 1
SNAKE_TRAIN conv 25 {"step": 25, "loss": 0.905890166759491, "file_name": "20140213024554Uh.jpeg", "gradient_norm": 0.0, "alpha": -0.012507895939052105, "scale": 64.0} elapsed_s 12
SNAKE_TRAIN conv 50 {"step": 50, "loss": 16.517837524414062, "file_name": "20110128174814Mh.jpeg", "gradient_norm": 0.3440485894680023, "alpha": -0.016654638573527336, "scale": 64.0} elapsed_s 22
SNAKE_TRAIN conv 75 {"step": 75, "loss": 13.884500503540039, "file_name": "20170806133710Th.jpeg", "gradient_norm": 1.0081850290298462, "alpha": -0.023489445447921753, "scale": 64.0} elapsed_s 33
SNAKE_TRAIN conv 100 {"step": 100, "loss": 47.160980224609375, "file_

In [ ]:
# Matched all-scale control and snake arms, within the existing wall cap.
assert summary['status']=='complete' and len(summary['arms'])==2
assert time.monotonic()-PILOT_STARTED+2400<PILOT_WALL_CAP
summary['status']='all_scale_extension_running'
# Separate feasibility probe; it makes no optimizer updates and does not alter arm inputs.
create_arm('snake',(0,1,2));model.train();model.backbone.eval()
for adapter in [model.sem_seg_head.snake_mask_adapter,*model.sem_seg_head.snake_scale_adapters.values()]:
    adapter.alpha.data.fill_(.01)
CROP_SIZE=2048;rng=random.Random(20261017);torch.cuda.reset_peak_memory_stats()
try:
    full_item,_=sample_training()
    with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):
        full_losses=model([full_item]);full_loss=sum(full_losses.values())
    with torch.enable_grad():full_loss.backward()
    assert torch.isfinite(full_loss)
    assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
    full_probe=dict(crop=2048,base_frozen=True,mode='snake_mask_plus_all_scales',
                    peak_vram_gib=torch.cuda.max_memory_allocated()/1024**3,optimizer_updates=0,status='PASS')
    del full_item,full_losses,full_loss
except torch.cuda.OutOfMemoryError:
    full_probe=dict(crop=2048,base_frozen=True,status='OOM',optimizer_updates=0)
    for key in ('full_item','full_losses','full_loss'):
        globals().pop(key,None)
finally:
    summary['preflights'].append(full_probe)
    print('SNAKE_FULL2048_PREFLIGHT',json.dumps(full_probe),flush=True)
    CROP_SIZE=1024;detach_adapters()
for mode in ('conv', 'snake'):
    assert time.monotonic() - PILOT_STARTED + 1200 < PILOT_WALL_CAP, 'Not enough time for another pilot arm'
    config = create_arm(mode, (0, 1, 2))
    model.train()
    model.backbone.eval()
    params = [p for p in model.parameters() if p.requires_grad]
    gate = [p for n, p in model.named_parameters() if p.requires_grad and n.endswith('.alpha')]
    others = [p for n, p in model.named_parameters() if p.requires_grad and (not n.endswith('.alpha'))]
    optimizer = torch.optim.AdamW([dict(params=others, lr=0.0001), dict(params=gate, lr=0.001)], weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', init_scale=64.0)
    rng = random.Random(20261018)
    torch.manual_seed(20261018)
    history = []
    updates = 0
    arm_started = time.monotonic()
    torch.cuda.reset_peak_memory_stats()
    for step in range(1, 201):
        assert time.monotonic() - PILOT_STARTED < PILOT_WALL_CAP - 900
        item, filename = sample_training()
        optimizer.zero_grad(set_to_none=True)
        with torch.enable_grad(), torch.autocast('cuda', dtype=torch.float16):
            losses = model([item])
            loss = sum(losses.values())
        assert torch.isfinite(loss), 'Nonfinite loss'
        previous_scale = scaler.get_scale()
        with torch.enable_grad():
            scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        updates += int(scaler.get_scale() >= previous_scale)
        history.append(dict(step=step, loss=float(loss.detach()), file_name=filename, gradient_norm=float(norm), alpha=float(gate[0].detach()), scale=scaler.get_scale()))
        del losses, loss, item
        if step == 1 or step % 25 == 0:
            print('SNAKE_TRAIN', mode, step, json.dumps(history[-1]), 'elapsed_s', round(time.monotonic() - arm_started), flush=True)
        if step % 50 == 0:
            checkpoint(OUT / (mode + '_all_scales_last.pth'), config, step, optimizer, scaler, rng, history)
    assert all((torch.equal(v.detach().cpu(), base_weights[k]) for k, v in model.state_dict().items() if '.snake_' not in k)), 'Frozen base changed'
    train_peak = torch.cuda.max_memory_allocated() / 1024 ** 3
    model.zero_grad(set_to_none=True)
    metrics = evaluate_arm(mode + '_all_scales')
    checkpoint(OUT / (mode + '_all_scales_last.pth'), config, 200, optimizer, scaler, rng, history, metrics)
    result = dict(mode=mode + '_all_scales', config=config, training_steps=200, optimizer_updates=updates, trainable_parameters=sum((p.numel() for p in params)), training_peak_vram_gib=train_peak, calibration=metrics, elapsed_seconds=round(time.monotonic() - arm_started))
    summary['arms'].append(result)
    (OUT / (mode + '_all_scales_history.json')).write_text(json.dumps(history))
    (OUT / 'pilot_summary.json').write_text(json.dumps(summary, indent=2))
    print('SNAKE_ARM_COMPLETE', json.dumps(result), flush=True)
    del optimizer, scaler, params, gate, others
    detach_adapters()
summary['status']='complete_four_arms'
summary['elapsed_seconds']=round(time.monotonic()-PILOT_STARTED)
(OUT/'pilot_summary.json').write_text(json.dumps(summary,indent=2))
with zipfile.ZipFile('/kaggle/working/snake_pilot_20261017.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.iterdir():archive.write(path,path.name)
print('SNAKE_FOUR_ARM_COMPLETE',json.dumps(summary),flush=True)


In [1]:
import os, sys, time, json, shutil, subprocess
from pathlib import Path

RUN_STARTED = time.monotonic()
import zipfile
archives = list(Path('/kaggle/input').rglob('solar_payload.bin'))
assert len(archives) == 1, archives
ASSETS = Path('/kaggle/working/solar_assets')
ASSETS.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    archive.extractall(ASSETS)
INPUTS = ASSETS
RUNTIME = Path('/kaggle/working/solar_m2f_runtime')
RUNTIME.mkdir(exist_ok=True)
for name in ['Mask2Former', 'detectron2']:
    target = RUNTIME / name
    if not target.exists():
        shutil.copytree(ASSETS / 'vendor' / name, target)
print('ASSETS:', ASSETS, flush=True)
print('Python:', sys.version, flush=True)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, flush=True)
assert torch.cuda.is_available(), 'A free Kaggle GPU is required; no CPU full run.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ninja', 'pycocotools', 'timm==0.9.16', 'fvcore==0.1.5.post20221221',
    'iopath==0.1.9', 'yacs', 'omegaconf>=2.1,<2.4', 'hydra-core',
    'termcolor', 'tabulate', 'cloudpickle', 'tensorboard', 'shapely',
    'scipy', 'submitit', 'black', 'packaging'], check=True, timeout=900)
# The Mask2Former R50 path needs no Detectron2 CUDA-specific operators.
# Build its normal CPU extension; deformable attention uses the existing
# project's differentiable PyTorch reference implementation during inference.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', MAX_JOBS='2')
log_path = Path('/kaggle/working/detectron2_install.log')
print('Building Detectron2 CPU extension; log:', log_path, flush=True)
with log_path.open('w') as log:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install',
        '--no-build-isolation', '--no-deps', str(RUNTIME / 'detectron2')],
        env=env, stdout=log, stderr=subprocess.STDOUT, timeout=1800)
print(log_path.read_text()[-8000:], flush=True)
assert result.returncode == 0, 'See detectron2_install.log'
sys.path[:0] = [str(ASSETS), str(RUNTIME / 'Mask2Former')]
from solar_runtime import install_msda_backend
BACKEND = install_msda_backend('torch')
import detectron2, mask2former
print('Runtime ready:', detectron2.__version__, BACKEND, flush=True)
print('SETUP_SECONDS:', round(time.monotonic() - RUN_STARTED), flush=True)

import csv, gc, hashlib, itertools, json, time, zipfile
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from pycocotools import mask as mu
from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.modeling import build_model
from detectron2.data import detection_utils as utils
from mask2former import add_maskformer2_config
from solar_inference import install_chunked_instance_inference
from solar_evaluation import mask_rle, pq_counts, summarize_counts

OUT = Path('/kaggle/working/snake2048_controls_20261018')
OUT.mkdir(exist_ok=True)
LIMIT_SECONDS = 5400
torch.set_grad_enabled(False)
torch.manual_seed(42)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
provenance = json.loads((ASSETS / 'provenance.json').read_text())
weight = ASSETS / 'model_best.pth'
assert hashlib.sha256(weight.read_bytes()).hexdigest() == provenance['checkpoint_sha256']
cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
cfg.set_new_allowed(True)
cfg.merge_from_file(str(ASSETS / 'config.yaml'))
cfg.MODEL.DEVICE = 'cuda'
cfg.DATASETS.TRAIN = ('solar_calibration_metadata',)
cfg.DATASETS.TEST = ()
model = build_model(cfg)
checkpoint = torch.load(weight, map_location='cpu', weights_only=True)
assert checkpoint['iteration'] == 36999
model.load_state_dict(checkpoint['model'], strict=True)
del checkpoint
gc.collect()
model.requires_grad_(False).eval()
install_chunked_instance_inference(model, .15)
records = json.loads((ASSETS / 'validation_records.json').read_text())
dates = sorted({r['file_name'][:8] for r in records},
               key=lambda x: hashlib.sha256(('calibration-v1-' + x).encode()).hexdigest())
cal_dates = set(dates[:len(dates)//2])
part = {r['file_name']: ('calibration' if r['file_name'][:8] in cal_dates else 'holdout') for r in records}
assert {r['file_name'][:8] for r in records if part[r['file_name']]=='calibration'}.isdisjoint(
       {r['file_name'][:8] for r in records if part[r['file_name']]=='holdout'})
(OUT/'split.json').write_text(json.dumps(part, indent=2))
root = next(ASSETS.rglob('test_images')).parent.parent
train_dir, test_dir = root/'train/train_images', root/'test/test_images'
raw = json.loads((root/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
gt = defaultdict(list)
for a in raw['annotations']:
    gt[a['image_id']].append(a)
VIEWS = ('native','hflip','vflip','rot180')

def encode(mask):
    r = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    r['counts'] = r['counts'].decode('ascii')
    return r

def candidates(path, view):
    assert time.monotonic()-RUN_STARTED < LIMIT_SECONDS, 'Calibration time cap reached'
    assert not model.training and not torch.is_grad_enabled()
    image = utils.read_image(str(path), format=cfg.INPUT.FORMAT)
    assert image.shape[:2] == (2048,2048)
    if view in ('hflip','rot180'): image = image[:,::-1]
    if view in ('vflip','rot180'): image = image[::-1]
    x = torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
    with torch.inference_mode():
        instances = model([{'image':x,'height':2048,'width':2048}])[0]['instances']
    result = []
    for idx in instances.scores.argsort(descending=True).tolist():
        mask = instances.pred_masks[idx].numpy()
        if view in ('hflip','rot180'): mask = mask[:,::-1]
        if view in ('vflip','rot180'): mask = mask[::-1]
        if mask.any(): result.append(dict(rle=encode(mask),score=float(instances.scores[idx]),view=view))
    del x, instances
    return result

def collect(directory, names, views, filename):
    path = OUT/filename
    cache = json.loads(path.read_text()) if path.exists() else {}
    started=time.monotonic()
    for i,name in enumerate(names,1):
        cache.setdefault(name,{})
        for view in views:
            if view not in cache[name]:
                cache[name][view] = candidates(directory/name,view)
                temp=path.with_suffix('.tmp');temp.write_text(json.dumps(cache));temp.replace(path)
        if i%10==0 or i==len(names):
            print('CANDIDATES',filename,i,'/',len(names),'elapsed_s',round(time.monotonic()-started),flush=True)
    return cache

def consensus(per_view):
    pool=[p for v in VIEWS for p in per_view[v]]
    if not pool: return []
    iou=mu.iou([p['rle'] for p in pool],[p['rle'] for p in pool],[0]*len(pool))
    order=sorted(range(len(pool)),key=lambda i:pool[i]['score'],reverse=True)
    used=set();result=[]
    for i in order:
        if i in used: continue
        matched=[i]
        for v in VIEWS:
            if v==pool[i]['view']: continue
            options=[j for j in order if j not in used and pool[j]['view']==v and iou[i,j]>.5]
            if options: matched.append(max(options,key=lambda j:iou[i,j]))
        used.update(matched)
        if len(matched)>=2:
            result.append(dict(rle=pool[i]['rle'],score=sum(pool[j]['score'] for j in matched)/len(matched),
                               support=len(matched)))
    return result

def disjoint(pool):
    occupied=np.zeros((2048,2048),bool); result=[]
    for p in sorted(pool,key=lambda p:p['score'],reverse=True):
        mask=mu.decode(p['rle']).astype(bool)
        mask &= ~occupied
        if not mask.any(): continue
        occupied |= mask
        rle=encode(mask)
        result.append(dict(rle=rle,score=p['score'],area=int(mu.area(rle))))
    return result




"""Bottleneck residual adapters inspired by Qi et al., DSConv (ICCV 2023).

Reference: https://arxiv.org/abs/2307.08388 (Sec. 3.1).
This is an independent 2D implementation, not a reproduction of DSCNet:
we use learned dense kernel weights, center-out cumulative transverse offsets,
bilinear sampling, and a zero-initialized residual gate around M2F features.
No DSCNet fusion strategy, topology loss, or medical data is included.
"""
import math

import torch
from torch import nn
from torch.nn import functional as F


class SnakeKernel2D(nn.Module):
    """Two connected K-point sampling paths, initialized as horizontal/vertical."""

    def __init__(self, channels, kernel_size=9, max_step=1., dynamic=True):
        super().__init__()
        if kernel_size < 3 or kernel_size % 2 != 1:
            raise ValueError('kernel_size must be odd and >= 3')
        self.channels = channels
        self.kernel_size = kernel_size
        self.max_step = float(max_step)
        self.dynamic = dynamic
        self.offset = nn.Conv2d(channels, 2 * (kernel_size - 1), 3, padding=1)
        nn.init.zeros_(self.offset.weight)
        nn.init.zeros_(self.offset.bias)
        self.weight = nn.Parameter(torch.empty(2, channels, channels, kernel_size))
        nn.init.normal_(self.weight, std=1. / math.sqrt(channels * kernel_size))
        if not dynamic:
            self.offset.requires_grad_(False)

    def transverse_offsets(self, x):
        b, _, h, w = x.shape
        k, radius = self.kernel_size, self.kernel_size // 2
        if not self.dynamic:
            return x.new_zeros((b, 2, k, h, w))
        steps = self.max_step * torch.tanh(self.offset(x))
        steps = steps.view(b, 2, k - 1, h, w)
        # On either side, steps are stored from the center towards the edge.
        left = steps[:, :, :radius].cumsum(2).flip(2)
        right = steps[:, :, radius:].cumsum(2)
        center = steps[:, :, :1] * 0.
        return torch.cat((left, center, right), dim=2)

    def forward(self, x):
        # Coordinates/sampling use FP32 for CUDA AMP compatibility and stability.
        # The inner 1x1 contractions may still be autocast by the caller.
        x = x.float()
        b, _, h, w = x.shape
        if min(h, w) <= 1:
            raise ValueError('Snake sampling requires H,W > 1')
        offsets = self.transverse_offsets(x).float()
        yy, xx = torch.meshgrid(torch.arange(h, device=x.device, dtype=x.dtype),
                                torch.arange(w, device=x.device, dtype=x.dtype), indexing='ij')
        output = []
        radius = self.kernel_size // 2
        for axis in range(2):
            value = None
            for idx in range(self.kernel_size):
                fixed = idx - radius
                if axis == 0:
                    gx, gy = xx + fixed, yy + offsets[:, axis, idx]
                else:
                    gx, gy = xx + offsets[:, axis, idx], yy + fixed
                gx = gx.expand(b, h, w)
                gy = gy.expand(b, h, w)
                grid = torch.stack((2 * gx / (w - 1) - 1, 2 * gy / (h - 1) - 1), dim=-1)
                # Sequential samples avoid a materialized B*C*K*H*W tensor.
                sampled = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True)
                weighted = F.conv2d(sampled, self.weight[axis, :, :, idx, None, None])
                value = weighted if value is None else value + weighted
            output.append(value)
        return torch.cat(output, dim=1)


class ResidualFeatureAdapter(nn.Module):
    def __init__(self, channels=256, bottleneck=32, mode='snake', kernel_size=9):
        super().__init__()
        if mode not in ('snake', 'fixed_snake', 'conv'):
            raise ValueError(mode)
        self.mode = mode
        self.reduce = nn.Sequential(nn.Conv2d(channels, bottleneck, 1),
                                    nn.GroupNorm(math.gcd(8, bottleneck), bottleneck), nn.GELU())
        self.spatial = (nn.Conv2d(bottleneck, 2 * bottleneck, 3, padding=1)
                        if mode == 'conv' else SnakeKernel2D(bottleneck, kernel_size,
                                                             dynamic=mode == 'snake'))
        self.expand = nn.Conv2d(2 * bottleneck, channels, 1)
        self.alpha = nn.Parameter(torch.zeros(()))

    def forward(self, x):
        correction = self.expand(F.gelu(self.spatial(self.reduce(x))))
        return x + self.alpha * correction.to(x.dtype)


def attach_feature_adapters(model, *, mode='snake', mask_features=True, scales=(), bottleneck=32, kernel_size=9):
    """Attach after loading original weights; expanded checkpoints attach before load.

    One mask adapter is shared by all query-decoder prediction heads; scale 0/1/2
    address the existing coarse-to-fine 1/32,1/16,1/8 features. The hook changes
    predictor inputs only. It never convolves query tokens or class logits.
    """
    head = model.sem_seg_head
    if hasattr(head, '_snake_feature_handle'):
        raise ValueError('Adapters already installed; remove before reinstalling')
    selected = tuple(sorted(set(scales)))
    if any(level not in (0, 1, 2) for level in selected):
        raise ValueError('scales must be a subset of (0,1,2)')
    if not mask_features and not selected:
        raise ValueError('Select at least one feature site')
    device = next(model.parameters()).device
    head.snake_mask_adapter = (ResidualFeatureAdapter(mode=mode, bottleneck=bottleneck, kernel_size=kernel_size)
                               if mask_features else nn.Identity()).to(device)
    head.snake_scale_adapters = nn.ModuleDict({str(i): ResidualFeatureAdapter(mode=mode, bottleneck=bottleneck,
                                                                           kernel_size=kernel_size)
                                              for i in selected}).to(device)
    config = dict(mode=mode, mask_features=mask_features, scales=list(selected),
                  channels=256, bottleneck=bottleneck, kernel_size=kernel_size,
                  reference='https://arxiv.org/abs/2307.08388', implementation='independent_bottleneck_residual_v1')
    head._snake_adapter_config = config

    def adapt_inputs(_module, args):
        features, masks, *rest = args
        if len(features) != 3:
            raise ValueError('Expected the original three M2F spatial feature levels')
        enhanced = [head.snake_scale_adapters[str(i)](feature) if str(i) in head.snake_scale_adapters else feature
                    for i, feature in enumerate(features)]
        return (enhanced, head.snake_mask_adapter(masks), *rest)

    head._snake_feature_handle = head.predictor.register_forward_pre_hook(adapt_inputs)
    return config

import copy, random, warnings, traceback, datetime
from collections import OrderedDict
from detectron2.structures import Instances
warnings.filterwarnings('ignore', category=FutureWarning)
EXPERIMENT_START=time.monotonic()
# Conservative allowance: <=85min GPU wall, <=170 GPU-hours minutes if both T4s charged.
WALL_CAP_SECONDS=5100
TRAIN_SEED=20261018
INIT_SEED=20261017
SOURCE=Path('/kaggle/working/finetune_results/model_best_finetune.pth')
SOURCE_SHA='6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1'
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SOURCE_SHA
base=torch.load(SOURCE,map_location='cpu',weights_only=True)
base_weights=base['model'];del base
model.load_state_dict(base_weights,strict=True)
training_root=Path('/kaggle/working/training_images')
if not (training_root/'training_manifest.json').exists():
    archives=list(Path('/kaggle/input').rglob('training_images.bin'));assert len(archives)==1
    training_root.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:z.extractall(training_root)
manifest=json.loads((training_root/'training_manifest.json').read_text())
train_names=manifest['train_images'];assert len(train_names)==566 and not set(train_names)&set(part)
train_records=defaultdict(list)
for image in raw['images']:
    if image['file_name'] in set(train_names):train_records[image['file_name']].append(image['id'])
MASK_PREFIX='sem_seg_head.predictor.mask_embed.'
assert any(n.startswith(MASK_PREFIX) for n,_ in model.named_parameters())
def emit(marker,data):
    line=marker+' '+json.dumps(data,ensure_ascii=False)
    print(line,flush=True)
    with (OUT/'events.jsonl').open('a') as f:f.write(json.dumps(dict(marker=marker,data=data,utc=datetime.datetime.now(datetime.timezone.utc).isoformat()))+'\n')
def write_json(path,data):
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(data,indent=2));tmp.replace(path)
def pack_outputs():
    dst=OUT.with_suffix('.zip');tmp=dst.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in sorted(OUT.iterdir()):
            if f.is_file() and f.suffix!='.tmp':z.write(f,f.name)
    tmp.replace(dst)
def cleanup():
    h=model.sem_seg_head
    if hasattr(h,'_snake_feature_handle'):
        h._snake_feature_handle.remove();del h._snake_feature_handle
        del h.snake_mask_adapter,h.snake_scale_adapters,h._snake_adapter_config
    model.zero_grad(set_to_none=True)
    gc.collect();torch.cuda.empty_cache()
def create_arm(mode):
    cleanup();model.load_state_dict(base_weights,strict=True);model.requires_grad_(False)
    torch.manual_seed(INIT_SEED)
    config=dict(mode='none',mask_features=False,scales=[])
    if mode!='head_only':config=attach_feature_adapters(model,mode=mode,mask_features=True,scales=(),bottleneck=32,kernel_size=9)
    model.sem_seg_head.predictor.mask_embed.requires_grad_(True)
    return config
def optimizer_for_arm():
    groups=[]
    for label,lr,predicate in [('mask_embed',1e-6,lambda n:n.startswith(MASK_PREFIX)),('gate',1e-3,lambda n:n.endswith('.alpha')),('adapter',1e-4,lambda n:'.snake_' in n and not n.endswith('.alpha'))]:
        params=[p for n,p in model.named_parameters() if p.requires_grad and predicate(n)]
        if params:groups.append(dict(params=params,lr=lr,group_name=label))
    params=[p for p in model.parameters() if p.requires_grad]
    assert len(params)==sum(len(g['params']) for g in groups)
    return torch.optim.AdamW(groups,weight_decay=.01),params
schedule=[];schedule_rng=random.Random(TRAIN_SEED)
for step in range(600):
    name=schedule_rng.choice(train_names);sid=schedule_rng.choice(train_records[name])
    schedule.append(dict(file_name=name,source_image_id=sid,hflip=schedule_rng.random()<.5,vflip=schedule_rng.random()<.5))
write_json(OUT/'training_schedule.json',schedule)
def training_item(spec):
    image=utils.read_image(str(training_root/'train_images'/spec['file_name']),format=cfg.INPUT.FORMAT)
    assert image.shape[:2]==(2048,2048)
    masks=[mu.decode(mask_rle(a['segmentation'],2048,2048)) for a in gt[spec['source_image_id']]]
    masks=np.stack(masks) if masks else np.zeros((0,2048,2048),np.uint8)
    if spec['hflip']:image=image[:,::-1];masks=masks[:,:,::-1]
    if spec['vflip']:image=image[::-1];masks=masks[:,::-1,:]
    target=Instances((2048,2048));target.gt_classes=torch.zeros(len(masks),dtype=torch.int64)
    target.gt_masks=torch.from_numpy(np.ascontiguousarray(masks)).bool()
    return dict(image=torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1))),instances=target,height=2048,width=2048)
def save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history,metrics=None):
    state=model.state_dict()
    delta=OrderedDict((k,v.detach().cpu()) for k,v in state.items() if '.snake_' in k or k.startswith(MASK_PREFIX))
    delta._metadata=copy.deepcopy(state._metadata)
    data=dict(format='mask_embed_and_adapter_delta_v1',delta=delta,adapter_config=config,arm=mode,
              source_checkpoint_sha256=SOURCE_SHA,source_checkpoint=str(SOURCE),effective_updates=updates,attempts=attempts,
              optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),python_rng=random.getstate(),numpy_rng=np.random.get_state(),
              torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all(),history=history,metrics=metrics,
              schedule_sha256=hashlib.sha256(json.dumps(schedule).encode()).hexdigest(),input_size=2048,
              trainable_names=[n for n,p in model.named_parameters() if p.requires_grad],
              recovery='Load original base, attach adapter_config when mode!=none, load delta strict=False with exact allowed missing keys, restore optimizer/scaler/RNG, resume schedule[effective_updates:].')
    path=OUT/(mode+'_last.pth');tmp=path.with_suffix('.tmp');torch.save(data,tmp);tmp.replace(path)
    write_json(OUT/(mode+'_history.json'),history)
def verify_frozen():
    state=model.state_dict()
    assert all(torch.equal(v.detach().cpu(),base_weights[k]) for k,v in state.items() if '.snake_' not in k and not k.startswith(MASK_PREFIX)), 'Unexpected frozen-base mutation'
    return dict(frozen_parameters_identical=True,mask_embed_changed=any(not torch.equal(v.detach().cpu(),base_weights[k]) for k,v in state.items() if k.startswith(MASK_PREFIX)),
                offset_nonzero={n:int(torch.count_nonzero(p.detach()).item()) for n,p in model.named_parameters() if '.spatial.offset.weight' in n},
                gates={n:float(p.detach()) for n,p in model.named_parameters() if n.endswith('.alpha')})
def evaluate_arm(mode):
    model.eval();cache={};selected=sorted(n for n in part if part[n]=='calibration');assert len(selected)==70
    with torch.inference_mode():
        for idx,name in enumerate(selected,1):
            assert time.monotonic()-EXPERIMENT_START<WALL_CAP_SECONDS-120,'Evaluation wall cap reached'
            cache[name]={v:candidates(train_dir/name,v) for v in VIEWS}
            write_json(OUT/(mode+'_candidates.json'),cache)
            if idx%10==0:emit('FULL2048_EVAL_PROGRESS',dict(arm=mode,images=idx,total=70,elapsed_seconds=round(time.monotonic()-EXPERIMENT_START)))
    pred={name:[p['rle'] for p in disjoint(consensus(views)) if p['score']>=.9 and p['area']>=256] for name,views in cache.items()}
    counts=[]
    for r in records:
        name=r['file_name']
        if name not in pred:continue
        truth=[mask_rle(a['segmentation'],2048,2048) for a in gt[r['source_image_id']]]
        ps=pred[name];mat=mu.iou(truth,ps,[0]*len(ps)) if ps else np.zeros((len(truth),0))
        counts.append(pq_counts(mat))
    return summarize_counts(counts)
summary=dict(experiment='shared_mask_adapter_plus_mask_embed_full2048',source_sha256=SOURCE_SHA,input_size=2048,
             input_policy='whole native2048, no crop/no downsample, matched H/V flips',planned_effective_updates=600,
             arms=[],preflights=[],baseline_calibration_pq=.4563696594893772,baseline_public_score=.36,
             mask_embed_lr=1e-6,adapter_lr=1e-4,gate_lr=1e-3,weight_decay=.01,grad_clip=1.,
             train_seed=TRAIN_SEED,init_seed=INIT_SEED,schedule_sha256=hashlib.sha256(json.dumps(schedule).encode()).hexdigest(),
             calibration_images=70,training_images=566,review_evaluated=False,submitted=False,
             evaluation='full2048FP32,4views,2-viewIoU>.5 consensus,mean score>=.90,area>=256,greedy nonoverlap',
             wall_cap_seconds=WALL_CAP_SECONDS,conservative_max_two_device_quota_hours=(WALL_CAP_SECONDS+300)/1800,
             status='preflight_running',device=torch.cuda.get_device_name(0),visible_devices=torch.cuda.device_count())
write_json(OUT/'summary.json',summary);emit('FULL2048_START',summary)
try:
    # Compare actual full-image inference at initialization, before any updates.
    model.eval();torch.set_grad_enabled(False)
    identity_name=sorted(n for n in part if part[n]=='calibration')[0]
    with torch.inference_mode():before=candidates(train_dir/identity_name,'native')
    create_arm('snake');model.eval()
    with torch.inference_mode():after=candidates(train_dir/identity_name,'native')
    assert before==after,'Zero-gate baseline identity failed'
    summary['actual_zero_gate_full2048_identity']=True
    # Two batches per arm, including densest training annotation; Adam state allocated.
    dense_name,dense_sid=max(((name,sid) for name in train_names for sid in train_records[name]),key=lambda item:len(gt[item[1]]))
    probe_specs=[schedule[0],dict(file_name=dense_name,source_image_id=dense_sid,hflip=False,vflip=False)]
    for mode in ('head_only','fixed_snake','snake'):
        config=create_arm(mode);model.train();model.backbone.eval()
        if mode!='head_only':model.sem_seg_head.snake_mask_adapter.alpha.data.fill_(.01)
        optimizer,params=optimizer_for_arm();scaler=torch.amp.GradScaler('cuda',init_scale=64.)
        torch.manual_seed(TRAIN_SEED);torch.cuda.reset_peak_memory_stats();probe_started=time.monotonic();probe_losses=[]
        for spec in probe_specs:
            item=training_item(spec);optimizer.zero_grad(set_to_none=True)
            with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):losses=model([item]);loss=sum(losses.values())
            assert torch.isfinite(loss),'Nonfinite preflight loss'
            with torch.enable_grad():scaler.scale(loss).backward()
            scaler.unscale_(optimizer);assert all(p.grad is None or torch.isfinite(p.grad).all() for p in params)
            torch.nn.utils.clip_grad_norm_(params,1.);scaler.step(optimizer);scaler.update();torch.cuda.synchronize()
            probe_losses.append(float(loss.detach()));del item,losses,loss
        probe=dict(arm=mode,status='PASS',input_size=2048,max_gt_instances=len(gt[dense_sid]),losses=probe_losses,
                   peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3,peak_reserved_gib=torch.cuda.max_memory_reserved()/1024**3,
                   free_gib=torch.cuda.mem_get_info()[0]/1024**3,probe_optimizer_steps=2,discarded_before_training=True,
                   elapsed_seconds=time.monotonic()-probe_started,trainable_parameters=sum(p.numel() for p in params))
        summary['preflights'].append(probe);write_json(OUT/'summary.json',summary);emit('FULL2048_PREFLIGHT_PASS',probe)
        assert probe['peak_reserved_gib']<.92*torch.cuda.get_device_properties(0).total_memory/1024**3,'Insufficient memory margin'
        del optimizer,scaler,params;cleanup()
    # Same effective update budget for all arms, chosen by preflight throughput only.
    worst=max(p['elapsed_seconds']/2 for p in summary['preflights'])
    available=WALL_CAP_SECONDS-(time.monotonic()-EXPERIMENT_START)-2100
    common_steps=min(600,int(available/(3*worst*1.20))//50*50)
    assert common_steps>=200,'Not enough bounded time for matched three-arm training'
    summary['effective_updates_per_arm']=common_steps;summary['status']='training';emit('FULL2048_MATCHED_BUDGET',dict(effective_updates=common_steps,estimated_worst_seconds_per_update=worst))
    for mode in ('head_only','fixed_snake','snake'):
        config=create_arm(mode);model.train();model.backbone.eval();torch.manual_seed(TRAIN_SEED);np.random.seed(TRAIN_SEED);random.seed(TRAIN_SEED)
        optimizer,params=optimizer_for_arm();scaler=torch.amp.GradScaler('cuda',init_scale=64.)
        updates=0;attempts=0;history=[];arm_started=time.monotonic();torch.cuda.reset_peak_memory_stats()
        while updates<common_steps:
            assert time.monotonic()-EXPERIMENT_START<WALL_CAP_SECONDS-900,'Training wall cap reached'
            assert attempts<common_steps+50,'Excessive AMP retries'
            spec=schedule[updates];item=training_item(spec);optimizer.zero_grad(set_to_none=True);attempts+=1
            with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):losses=model([item]);loss=sum(losses.values())
            assert torch.isfinite(loss),'Nonfinite training loss'
            old_scale=scaler.get_scale()
            with torch.enable_grad():scaler.scale(loss).backward()
            scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(params,1.);scaler.step(optimizer);scaler.update()
            accepted=scaler.get_scale()>=old_scale;updates+=int(accepted)
            entry=dict(attempt=attempts,effective_update=updates,optimizer_updated=bool(accepted),loss=float(loss.detach()),gradient_norm=float(norm),scale=scaler.get_scale(),**spec)
            if mode!='head_only':entry['alpha']=float(model.sem_seg_head.snake_mask_adapter.alpha.detach())
            history.append(entry);del item,losses,loss
            if attempts==1 or attempts%25==0:emit('FULL2048_TRAIN',dict(arm=mode,**entry,elapsed_seconds=round(time.monotonic()-arm_started)))
            if accepted and updates%50==0:save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history)
        training_peak=torch.cuda.max_memory_allocated()/1024**3
        checks=verify_frozen();save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history)
        model.zero_grad(set_to_none=True);metrics=evaluate_arm(mode)
        save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history,metrics)
        result=dict(arm=mode,config=config,effective_updates=updates,attempts=attempts,calibration=metrics,
                    training_peak_allocated_gib=training_peak,trainable_parameters=sum(p.numel() for p in params),
                    integrity=checks,elapsed_seconds=round(time.monotonic()-arm_started))
        summary['arms'].append(result);write_json(OUT/'summary.json',summary);emit('FULL2048_ARM_COMPLETE',result)
        del optimizer,scaler,params;cleanup();pack_outputs()
    summary['status']='complete_three_arms'
except Exception as error:
    summary['status']='error';summary['error_type']=type(error).__name__;summary['error']=str(error);traceback.print_exc()
    emit('FULL2048_ERROR',dict(type=type(error).__name__,message=str(error)))
    if 'optimizer' in globals() and 'history' in globals():
        save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history)
finally:
    summary['elapsed_seconds']=round(time.monotonic()-EXPERIMENT_START)
    summary['completed_at_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat()
    write_json(OUT/'summary.json',summary);emit('FULL2048_COMPLETE',summary)
    model.eval();model.zero_grad(set_to_none=True);gc.collect();torch.cuda.empty_cache();pack_outputs()
    print('FULL2048_ARCHIVE_READY',str(OUT.with_suffix('.zip')),flush=True)


ASSETS: /kaggle/working/solar_assets
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu128 CUDA: 12.8
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 6.4 MB/s eta 0:00:00
Building Detectron2 CPU extension; log: /kaggle/working/detectron2_install.log
Processing ./solar_m2f_runtime/detectron2
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for detectron2: filename=detectron2-0.6-cp313-cp313-linux_x86_64.whl size=6675511 sha256=d2683fbc3fb5b000cc4606d13a604a843206bed17079dfbfb8440d84fca406a2
  Stored in directory: /tmp/pip-ephem-wheel-cache-yypu8g2b/wheels/8e/7f

/kaggle/working/solar_assets/solar_runtime.py:48: UserWarning: Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.
  warnings.warn('Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.')
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/ops/modules/ms_deform_attn.py:87: SyntaxWarning: invalid escape sequence '\s'
  :param input_flatten               (N, \sum_{l=0}^{L-1} H_l \cdot W_l, C)
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/msdeformattn.py:314: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


Runtime ready: 0.6 pytorch_reference
SETUP_SECONDS: 127
FULL2048_START {"experiment": "shared_mask_adapter_plus_mask_embed_full2048", "source_sha256": "6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1", "input_size": 2048, "input_policy": "whole native2048, no crop/no downsample, matched H/V flips", "planned_effective_updates": 600, "arms": [], "preflights": [], "baseline_calibration_pq": 0.4563696594893772, "baseline_public_score": 0.36, "mask_embed_lr": 1e-06, "adapter_lr": 0.0001, "gate_lr": 0.001, "weight_decay": 0.01, "grad_clip": 1.0, "train_seed": 20261018, "init_seed": 20261017, "schedule_sha256": "36908ae7ad379c2b39d253ad92d0c35ec349ec96051e0f6a5ca46c447c7ef27c", "calibration_images": 70, "training_images": 566, "review_evaluated": false, "submitted": false, "evaluation": "full2048FP32,4views,2-viewIoU>.5 consensus,mean score>=.90,area>=256,greedy nonoverlap", "wall_cap_seconds": 5100, "conservative_max_two_device_quota_hours": 3.0, "status": "preflight_running

/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


FULL2048_PREFLIGHT_PASS {"arm": "head_only", "status": "PASS", "input_size": 2048, "max_gt_instances": 24, "losses": [12.719954490661621, 33.86829376220703], "peak_allocated_gib": 4.399165153503418, "peak_reserved_gib": 5.49609375, "free_gib": 8.92266845703125, "probe_optimizer_steps": 2, "discarded_before_training": true, "elapsed_seconds": 5.709185894000029, "trainable_parameters": 197376}
FULL2048_PREFLIGHT_PASS {"arm": "fixed_snake", "status": "PASS", "input_size": 2048, "max_gt_instances": 24, "losses": [12.729682922363281, 33.97929763793945], "peak_allocated_gib": 4.399827480316162, "peak_reserved_gib": 5.5078125, "free_gib": 8.90899658203125, "probe_optimizer_steps": 2, "discarded_before_training": true, "elapsed_seconds": 3.889609856999982, "trainable_parameters": 240737}
FULL2048_PREFLIGHT_PASS {"arm": "snake", "status": "PASS", "input_size": 2048, "max_gt_instances": 24, "losses": [12.729682922363281, 33.86872100830078], "peak_allocated_gib": 4.399827480316162, "peak_reserved

In [1]:
import os, sys, time, json, shutil, subprocess
from pathlib import Path

RUN_STARTED = time.monotonic()
import zipfile
archives = list(Path('/kaggle/input').rglob('solar_payload.bin'))
assert len(archives) == 1, archives
ASSETS = Path('/kaggle/working/solar_assets')
ASSETS.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    archive.extractall(ASSETS)
INPUTS = ASSETS
RUNTIME = Path('/kaggle/working/solar_m2f_runtime')
RUNTIME.mkdir(exist_ok=True)
for name in ['Mask2Former', 'detectron2']:
    target = RUNTIME / name
    if not target.exists():
        shutil.copytree(ASSETS / 'vendor' / name, target)
print('ASSETS:', ASSETS, flush=True)
print('Python:', sys.version, flush=True)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, flush=True)
assert torch.cuda.is_available(), 'A free Kaggle GPU is required; no CPU full run.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ninja', 'pycocotools', 'timm==0.9.16', 'fvcore==0.1.5.post20221221',
    'iopath==0.1.9', 'yacs', 'omegaconf>=2.1,<2.4', 'hydra-core',
    'termcolor', 'tabulate', 'cloudpickle', 'tensorboard', 'shapely',
    'scipy', 'submitit', 'black', 'packaging'], check=True, timeout=900)
# The Mask2Former R50 path needs no Detectron2 CUDA-specific operators.
# Build its normal CPU extension; deformable attention uses the existing
# project's differentiable PyTorch reference implementation during inference.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', MAX_JOBS='2')
log_path = Path('/kaggle/working/detectron2_install.log')
print('Building Detectron2 CPU extension; log:', log_path, flush=True)
with log_path.open('w') as log:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install',
        '--no-build-isolation', '--no-deps', str(RUNTIME / 'detectron2')],
        env=env, stdout=log, stderr=subprocess.STDOUT, timeout=1800)
print(log_path.read_text()[-8000:], flush=True)
assert result.returncode == 0, 'See detectron2_install.log'
sys.path[:0] = [str(ASSETS), str(RUNTIME / 'Mask2Former')]
from solar_runtime import install_msda_backend
BACKEND = install_msda_backend('torch')
import detectron2, mask2former
print('Runtime ready:', detectron2.__version__, BACKEND, flush=True)
print('SETUP_SECONDS:', round(time.monotonic() - RUN_STARTED), flush=True)

import csv, gc, hashlib, itertools, json, time, zipfile
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from pycocotools import mask as mu
from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.modeling import build_model
from detectron2.data import detection_utils as utils
from mask2former import add_maskformer2_config
from solar_inference import install_chunked_instance_inference
from solar_evaluation import mask_rle, pq_counts, summarize_counts

OUT = Path('/kaggle/working/snake2048_resume_20261018')
OUT.mkdir(exist_ok=True)
LIMIT_SECONDS = 11100
torch.set_grad_enabled(False)
torch.manual_seed(42)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
provenance = json.loads((ASSETS / 'provenance.json').read_text())
weight = ASSETS / 'model_best.pth'
assert hashlib.sha256(weight.read_bytes()).hexdigest() == provenance['checkpoint_sha256']
cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
cfg.set_new_allowed(True)
cfg.merge_from_file(str(ASSETS / 'config.yaml'))
cfg.MODEL.DEVICE = 'cuda'
cfg.DATASETS.TRAIN = ('solar_calibration_metadata',)
cfg.DATASETS.TEST = ()
model = build_model(cfg)
checkpoint = torch.load(weight, map_location='cpu', weights_only=True)
assert checkpoint['iteration'] == 36999
model.load_state_dict(checkpoint['model'], strict=True)
del checkpoint
gc.collect()
model.requires_grad_(False).eval()
install_chunked_instance_inference(model, .15)
records = json.loads((ASSETS / 'validation_records.json').read_text())
dates = sorted({r['file_name'][:8] for r in records},
               key=lambda x: hashlib.sha256(('calibration-v1-' + x).encode()).hexdigest())
cal_dates = set(dates[:len(dates)//2])
part = {r['file_name']: ('calibration' if r['file_name'][:8] in cal_dates else 'holdout') for r in records}
assert {r['file_name'][:8] for r in records if part[r['file_name']]=='calibration'}.isdisjoint(
       {r['file_name'][:8] for r in records if part[r['file_name']]=='holdout'})
(OUT/'split.json').write_text(json.dumps(part, indent=2))
root = next(ASSETS.rglob('test_images')).parent.parent
train_dir, test_dir = root/'train/train_images', root/'test/test_images'
raw = json.loads((root/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
gt = defaultdict(list)
for a in raw['annotations']:
    gt[a['image_id']].append(a)
VIEWS = ('native','hflip','vflip','rot180')

def encode(mask):
    r = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    r['counts'] = r['counts'].decode('ascii')
    return r

def candidates(path, view):
    assert time.monotonic()-RUN_STARTED < LIMIT_SECONDS, 'Calibration time cap reached'
    assert not model.training and not torch.is_grad_enabled()
    image = utils.read_image(str(path), format=cfg.INPUT.FORMAT)
    assert image.shape[:2] == (2048,2048)
    if view in ('hflip','rot180'): image = image[:,::-1]
    if view in ('vflip','rot180'): image = image[::-1]
    x = torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
    with torch.inference_mode():
        instances = model([{'image':x,'height':2048,'width':2048}])[0]['instances']
    result = []
    for idx in instances.scores.argsort(descending=True).tolist():
        mask = instances.pred_masks[idx].numpy()
        if view in ('hflip','rot180'): mask = mask[:,::-1]
        if view in ('vflip','rot180'): mask = mask[::-1]
        if mask.any(): result.append(dict(rle=encode(mask),score=float(instances.scores[idx]),view=view))
    del x, instances
    return result

def collect(directory, names, views, filename):
    path = OUT/filename
    cache = json.loads(path.read_text()) if path.exists() else {}
    started=time.monotonic()
    for i,name in enumerate(names,1):
        cache.setdefault(name,{})
        for view in views:
            if view not in cache[name]:
                cache[name][view] = candidates(directory/name,view)
                temp=path.with_suffix('.tmp');temp.write_text(json.dumps(cache));temp.replace(path)
        if i%10==0 or i==len(names):
            print('CANDIDATES',filename,i,'/',len(names),'elapsed_s',round(time.monotonic()-started),flush=True)
    return cache

def consensus(per_view):
    pool=[p for v in VIEWS for p in per_view[v]]
    if not pool: return []
    iou=mu.iou([p['rle'] for p in pool],[p['rle'] for p in pool],[0]*len(pool))
    order=sorted(range(len(pool)),key=lambda i:pool[i]['score'],reverse=True)
    used=set();result=[]
    for i in order:
        if i in used: continue
        matched=[i]
        for v in VIEWS:
            if v==pool[i]['view']: continue
            options=[j for j in order if j not in used and pool[j]['view']==v and iou[i,j]>.5]
            if options: matched.append(max(options,key=lambda j:iou[i,j]))
        used.update(matched)
        if len(matched)>=2:
            result.append(dict(rle=pool[i]['rle'],score=sum(pool[j]['score'] for j in matched)/len(matched),
                               support=len(matched)))
    return result

def disjoint(pool):
    occupied=np.zeros((2048,2048),bool); result=[]
    for p in sorted(pool,key=lambda p:p['score'],reverse=True):
        mask=mu.decode(p['rle']).astype(bool)
        mask &= ~occupied
        if not mask.any(): continue
        occupied |= mask
        rle=encode(mask)
        result.append(dict(rle=rle,score=p['score'],area=int(mu.area(rle))))
    return result





"""Bottleneck residual adapters inspired by Qi et al., DSConv (ICCV 2023).

Reference: https://arxiv.org/abs/2307.08388 (Sec. 3.1).
This is an independent 2D implementation, not a reproduction of DSCNet:
we use learned dense kernel weights, center-out cumulative transverse offsets,
bilinear sampling, and a zero-initialized residual gate around M2F features.
No DSCNet fusion strategy, topology loss, or medical data is included.
"""
import math

import torch
from torch import nn
from torch.nn import functional as F


class SnakeKernel2D(nn.Module):
    """Two connected K-point sampling paths, initialized as horizontal/vertical."""

    def __init__(self, channels, kernel_size=9, max_step=1., dynamic=True):
        super().__init__()
        if kernel_size < 3 or kernel_size % 2 != 1:
            raise ValueError('kernel_size must be odd and >= 3')
        self.channels = channels
        self.kernel_size = kernel_size
        self.max_step = float(max_step)
        self.dynamic = dynamic
        self.offset = nn.Conv2d(channels, 2 * (kernel_size - 1), 3, padding=1)
        nn.init.zeros_(self.offset.weight)
        nn.init.zeros_(self.offset.bias)
        self.weight = nn.Parameter(torch.empty(2, channels, channels, kernel_size))
        nn.init.normal_(self.weight, std=1. / math.sqrt(channels * kernel_size))
        if not dynamic:
            self.offset.requires_grad_(False)

    def transverse_offsets(self, x):
        b, _, h, w = x.shape
        k, radius = self.kernel_size, self.kernel_size // 2
        if not self.dynamic:
            return x.new_zeros((b, 2, k, h, w))
        steps = self.max_step * torch.tanh(self.offset(x))
        steps = steps.view(b, 2, k - 1, h, w)
        # On either side, steps are stored from the center towards the edge.
        left = steps[:, :, :radius].cumsum(2).flip(2)
        right = steps[:, :, radius:].cumsum(2)
        center = steps[:, :, :1] * 0.
        return torch.cat((left, center, right), dim=2)

    def forward(self, x):
        # Coordinates/sampling use FP32 for CUDA AMP compatibility and stability.
        # The inner 1x1 contractions may still be autocast by the caller.
        x = x.float()
        b, _, h, w = x.shape
        if min(h, w) <= 1:
            raise ValueError('Snake sampling requires H,W > 1')
        offsets = self.transverse_offsets(x).float()
        yy, xx = torch.meshgrid(torch.arange(h, device=x.device, dtype=x.dtype),
                                torch.arange(w, device=x.device, dtype=x.dtype), indexing='ij')
        output = []
        radius = self.kernel_size // 2
        for axis in range(2):
            value = None
            for idx in range(self.kernel_size):
                fixed = idx - radius
                if axis == 0:
                    gx, gy = xx + fixed, yy + offsets[:, axis, idx]
                else:
                    gx, gy = xx + offsets[:, axis, idx], yy + fixed
                gx = gx.expand(b, h, w)
                gy = gy.expand(b, h, w)
                grid = torch.stack((2 * gx / (w - 1) - 1, 2 * gy / (h - 1) - 1), dim=-1)
                # Sequential samples avoid a materialized B*C*K*H*W tensor.
                sampled = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True)
                weighted = F.conv2d(sampled, self.weight[axis, :, :, idx, None, None])
                value = weighted if value is None else value + weighted
            output.append(value)
        return torch.cat(output, dim=1)


class ResidualFeatureAdapter(nn.Module):
    def __init__(self, channels=256, bottleneck=32, mode='snake', kernel_size=9):
        super().__init__()
        if mode not in ('snake', 'fixed_snake', 'conv'):
            raise ValueError(mode)
        self.mode = mode
        self.reduce = nn.Sequential(nn.Conv2d(channels, bottleneck, 1),
                                    nn.GroupNorm(math.gcd(8, bottleneck), bottleneck), nn.GELU())
        self.spatial = (nn.Conv2d(bottleneck, 2 * bottleneck, 3, padding=1)
                        if mode == 'conv' else SnakeKernel2D(bottleneck, kernel_size,
                                                             dynamic=mode == 'snake'))
        self.expand = nn.Conv2d(2 * bottleneck, channels, 1)
        self.alpha = nn.Parameter(torch.zeros(()))

    def forward(self, x):
        correction = self.expand(F.gelu(self.spatial(self.reduce(x))))
        return x + self.alpha * correction.to(x.dtype)


def attach_feature_adapters(model, *, mode='snake', mask_features=True, scales=(), bottleneck=32, kernel_size=9):
    """Attach after loading original weights; expanded checkpoints attach before load.

    One mask adapter is shared by all query-decoder prediction heads; scale 0/1/2
    address the existing coarse-to-fine 1/32,1/16,1/8 features. The hook changes
    predictor inputs only. It never convolves query tokens or class logits.
    """
    head = model.sem_seg_head
    if hasattr(head, '_snake_feature_handle'):
        raise ValueError('Adapters already installed; remove before reinstalling')
    selected = tuple(sorted(set(scales)))
    if any(level not in (0, 1, 2) for level in selected):
        raise ValueError('scales must be a subset of (0,1,2)')
    if not mask_features and not selected:
        raise ValueError('Select at least one feature site')
    device = next(model.parameters()).device
    head.snake_mask_adapter = (ResidualFeatureAdapter(mode=mode, bottleneck=bottleneck, kernel_size=kernel_size)
                               if mask_features else nn.Identity()).to(device)
    head.snake_scale_adapters = nn.ModuleDict({str(i): ResidualFeatureAdapter(mode=mode, bottleneck=bottleneck,
                                                                           kernel_size=kernel_size)
                                              for i in selected}).to(device)
    config = dict(mode=mode, mask_features=mask_features, scales=list(selected),
                  channels=256, bottleneck=bottleneck, kernel_size=kernel_size,
                  reference='https://arxiv.org/abs/2307.08388', implementation='independent_bottleneck_residual_v1')
    head._snake_adapter_config = config

    def adapt_inputs(_module, args):
        features, masks, *rest = args
        if len(features) != 3:
            raise ValueError('Expected the original three M2F spatial feature levels')
        enhanced = [head.snake_scale_adapters[str(i)](feature) if str(i) in head.snake_scale_adapters else feature
                    for i, feature in enumerate(features)]
        return (enhanced, head.snake_mask_adapter(masks), *rest)

    head._snake_feature_handle = head.predictor.register_forward_pre_hook(adapt_inputs)
    return config

import copy, random, warnings, traceback, datetime
from collections import OrderedDict
from detectron2.structures import Instances
warnings.filterwarnings('ignore', category=FutureWarning)
EXPERIMENT_START=time.monotonic()
# Conservative allowance: <=85min GPU wall, <=170 GPU-hours minutes if both T4s charged.
WALL_CAP_SECONDS=10800
TRAIN_SEED=20261018
INIT_SEED=20261017
SOURCE=Path('/kaggle/working/finetune_results/model_best_finetune.pth')
SOURCE_SHA='6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1'
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SOURCE_SHA
base=torch.load(SOURCE,map_location='cpu',weights_only=True)
base_weights=base['model'];del base
model.load_state_dict(base_weights,strict=True)
training_root=Path('/kaggle/working/training_images')
if not (training_root/'training_manifest.json').exists():
    archives=list(Path('/kaggle/input').rglob('training_images.bin'));assert len(archives)==1
    training_root.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:z.extractall(training_root)
manifest=json.loads((training_root/'training_manifest.json').read_text())
train_names=manifest['train_images'];assert len(train_names)==566 and not set(train_names)&set(part)
train_records=defaultdict(list)
for image in raw['images']:
    if image['file_name'] in set(train_names):train_records[image['file_name']].append(image['id'])
MASK_PREFIX='sem_seg_head.predictor.mask_embed.'
assert any(n.startswith(MASK_PREFIX) for n,_ in model.named_parameters())
def emit(marker,data):
    line=marker+' '+json.dumps(data,ensure_ascii=False)
    print(line,flush=True)
    with (OUT/'events.jsonl').open('a') as f:f.write(json.dumps(dict(marker=marker,data=data,utc=datetime.datetime.now(datetime.timezone.utc).isoformat()))+'\n')
def write_json(path,data):
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(data,indent=2));tmp.replace(path)
def pack_outputs():
    dst=OUT.with_suffix('.zip');tmp=dst.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in sorted(OUT.iterdir()):
            if f.is_file() and f.suffix!='.tmp':z.write(f,f.name)
    tmp.replace(dst)
def cleanup():
    h=model.sem_seg_head
    if hasattr(h,'_snake_feature_handle'):
        h._snake_feature_handle.remove();del h._snake_feature_handle
        del h.snake_mask_adapter,h.snake_scale_adapters,h._snake_adapter_config
    model.zero_grad(set_to_none=True)
    gc.collect();torch.cuda.empty_cache()
def create_arm(mode):
    cleanup();model.load_state_dict(base_weights,strict=True);model.requires_grad_(False)
    torch.manual_seed(INIT_SEED)
    config=dict(mode='none',mask_features=False,scales=[])
    if mode!='head_only':config=attach_feature_adapters(model,mode=mode,mask_features=True,scales=(),bottleneck=32,kernel_size=9)
    model.sem_seg_head.predictor.mask_embed.requires_grad_(True)
    return config
def optimizer_for_arm():
    groups=[]
    for label,lr,predicate in [('mask_embed',1e-6,lambda n:n.startswith(MASK_PREFIX)),('gate',1e-3,lambda n:n.endswith('.alpha')),('adapter',1e-4,lambda n:'.snake_' in n and not n.endswith('.alpha'))]:
        params=[p for n,p in model.named_parameters() if p.requires_grad and predicate(n)]
        if params:groups.append(dict(params=params,lr=lr,group_name=label))
    params=[p for p in model.parameters() if p.requires_grad]
    assert len(params)==sum(len(g['params']) for g in groups)
    return torch.optim.AdamW(groups,weight_decay=.01),params
schedule=[];schedule_rng=random.Random(TRAIN_SEED)
for step in range(1200):
    name=schedule_rng.choice(train_names);sid=schedule_rng.choice(train_records[name])
    schedule.append(dict(file_name=name,source_image_id=sid,hflip=schedule_rng.random()<.5,vflip=schedule_rng.random()<.5))
write_json(OUT/'training_schedule.json',schedule)
def training_item(spec):
    image=utils.read_image(str(training_root/'train_images'/spec['file_name']),format=cfg.INPUT.FORMAT)
    assert image.shape[:2]==(2048,2048)
    masks=[mu.decode(mask_rle(a['segmentation'],2048,2048)) for a in gt[spec['source_image_id']]]
    masks=np.stack(masks) if masks else np.zeros((0,2048,2048),np.uint8)
    if spec['hflip']:image=image[:,::-1];masks=masks[:,:,::-1]
    if spec['vflip']:image=image[::-1];masks=masks[:,::-1,:]
    target=Instances((2048,2048));target.gt_classes=torch.zeros(len(masks),dtype=torch.int64)
    target.gt_masks=torch.from_numpy(np.ascontiguousarray(masks)).bool()
    return dict(image=torch.as_tensor(np.ascontiguousarray(image.transpose(2,0,1))),instances=target,height=2048,width=2048)
def save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history,metrics=None):
    state=model.state_dict()
    delta=OrderedDict((k,v.detach().cpu()) for k,v in state.items() if '.snake_' in k or k.startswith(MASK_PREFIX))
    delta._metadata=copy.deepcopy(state._metadata)
    data=dict(origin_step250_sha256=PARENT_SHA[mode],parent_schedule_sha256=PARENT_SCHEDULE_SHA,resume_input_checkpoint_sha256=RESUME_INPUT_SHA,format='mask_embed_and_adapter_delta_v1',delta=delta,adapter_config=config,arm=mode,
              source_checkpoint_sha256=SOURCE_SHA,source_checkpoint=str(SOURCE),effective_updates=updates,attempts=attempts,
              optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),python_rng=random.getstate(),numpy_rng=np.random.get_state(),
              torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all(),history=history,metrics=metrics,
              schedule_sha256=hashlib.sha256(json.dumps(schedule).encode()).hexdigest(),input_size=2048,
              trainable_names=[n for n,p in model.named_parameters() if p.requires_grad],
              recovery='Load original base, attach adapter_config when mode!=none, load delta strict=False with exact allowed missing keys, restore optimizer/scaler/RNG, resume schedule[effective_updates:].')
    path=OUT/(mode+'_last.pth');tmp=path.with_suffix('.tmp');torch.save(data,tmp);tmp.replace(path)
    write_json(OUT/(mode+'_history.json'),history)
def verify_frozen():
    state=model.state_dict()
    assert all(torch.equal(v.detach().cpu(),base_weights[k]) for k,v in state.items() if '.snake_' not in k and not k.startswith(MASK_PREFIX)), 'Unexpected frozen-base mutation'
    return dict(frozen_parameters_identical=True,mask_embed_changed=any(not torch.equal(v.detach().cpu(),base_weights[k]) for k,v in state.items() if k.startswith(MASK_PREFIX)),
                offset_nonzero={n:int(torch.count_nonzero(p.detach()).item()) for n,p in model.named_parameters() if '.spatial.offset.weight' in n},
                gates={n:float(p.detach()) for n,p in model.named_parameters() if n.endswith('.alpha')})
def evaluate_arm(mode):
    model.eval();cache_path=OUT/(mode+'_candidates.json');cache=json.loads(cache_path.read_text()) if cache_path.exists() else {};selected=sorted(n for n in part if part[n]=='calibration');assert len(selected)==70
    with torch.inference_mode():
        for idx,name in enumerate(selected,1):
            assert time.monotonic()-EXPERIMENT_START<WALL_CAP_SECONDS-120,'Evaluation wall cap reached'
            if name not in cache:cache[name]={v:candidates(train_dir/name,v) for v in VIEWS}
            write_json(OUT/(mode+'_candidates.json'),cache)
            if idx%10==0:emit('RESUME2048_EVAL_PROGRESS',dict(arm=mode,images=idx,total=70,elapsed_seconds=round(time.monotonic()-EXPERIMENT_START)))
    pred={name:[p['rle'] for p in disjoint(consensus(views)) if p['score']>=.9 and p['area']>=256] for name,views in cache.items()}
    counts=[]
    for r in records:
        name=r['file_name']
        if name not in pred:continue
        truth=[mask_rle(a['segmentation'],2048,2048) for a in gt[r['source_image_id']]]
        ps=pred[name];mat=mu.iou(truth,ps,[0]*len(ps)) if ps else np.zeros((len(truth),0))
        counts.append(pq_counts(mat))
    return summarize_counts(counts)

PARENT=Path('/kaggle/working/snake2048_controls_20261018')
PARENT_SHA={'head_only': '2666428a467377c8bbaaa4069c99e1a0965e65bc88ad0ba18ae5c9f4237de2a5', 'fixed_snake': '0f624da140600ccf94addc983dc5fc2482623df49251ebd601db4d9601d85a1f', 'snake': '0bb41ccd6abfb250a6f03c0313651bbfb3752bdac5b49cab5107ddc504ad2d21'}
parent_schedule=json.loads((PARENT/'training_schedule.json').read_text())
assert len(parent_schedule)==600 and schedule[:600]==parent_schedule
PARENT_SCHEDULE_SHA=hashlib.sha256(json.dumps(parent_schedule).encode()).hexdigest()
for mode,expected in PARENT_SHA.items():
    assert hashlib.sha256((PARENT/(mode+'_last.pth')).read_bytes()).hexdigest()==expected

def restore_arm(mode,path):
    global RESUME_INPUT_SHA
    RESUME_INPUT_SHA=hashlib.sha256(path.read_bytes()).hexdigest()
    cp=torch.load(path,map_location='cpu',weights_only=False)
    assert cp['source_checkpoint_sha256']==SOURCE_SHA and cp['arm']==mode and cp['input_size']==2048
    config=create_arm(mode)
    assert config==cp['adapter_config']
    assert hasattr(cp['delta'],'_metadata') and cp['delta']._metadata['sem_seg_head']['version']==2
    expected_missing=set(model.state_dict())-set(cp['delta'])
    report=model.load_state_dict(cp['delta'],strict=False)
    assert not report.unexpected_keys and set(report.missing_keys)==expected_missing
    assert all(torch.equal(model.state_dict()[k].detach().cpu(),v) for k,v in cp['delta'].items())
    assert [n for n,p in model.named_parameters() if p.requires_grad]==cp['trainable_names']
    for row in cp['history']:
        if not row['optimizer_updated']:continue
        spec={k:row[k] for k in ('file_name','source_image_id','hflip','vflip')}
        assert spec==schedule[row['effective_update']-1]
    optimizer,params=optimizer_for_arm();optimizer.load_state_dict(cp['optimizer'])
    scaler=torch.amp.GradScaler('cuda',init_scale=64.);scaler.load_state_dict(cp['scaler'])
    random.setstate(cp['python_rng']);np.random.set_state(cp['numpy_rng']);torch.set_rng_state(cp['torch_rng'])
    assert len(cp['cuda_rng'])==torch.cuda.device_count()
    torch.cuda.set_rng_state_all(cp['cuda_rng'])
    return cp,config,optimizer,scaler,params

summary_path=OUT/'summary.json'
if summary_path.exists():
    summary=json.loads(summary_path.read_text())
    assert summary['status'] not in ('complete','stopped_no_calibration_improvement'), 'This continuation already completed; do not rerun it.'
else:
    summary=dict(experiment='native2048_matched_resume250',source_sha256=SOURCE_SHA,
                 origin_step250_sha256=PARENT_SHA,parent_schedule_sha256=PARENT_SCHEDULE_SHA,
                 schedule_sha256=hashlib.sha256(json.dumps(schedule).encode()).hexdigest(),
                 input_size=2048,input_policy='whole native2048, no crop or downsample',
                 targets=[600,1200],rounds=[],preflights=[],status='restoring',
                 baseline_calibration_pq=.4563696594893772,baseline_public_score=.36,
                 parent_calibration_pq=dict(head_only=.45524382357028764,fixed_snake=.4541071485523339,snake=.4555638268539116),
                 trainable_original_module=MASK_PREFIX,training_images=566,calibration_images=70,
                 mask_embed_lr=1e-6,adapter_lr=1e-4,gate_lr=1e-3,weight_decay=.01,
                 wall_cap_seconds=WALL_CAP_SECONDS,review_evaluated=False,submitted=False,
                 continuation_policy='Run all three arms to600. Continue all three to1200 only if at least one600 PQ improves its own250 PQ by>.001. Selection uses calibration only. No CSV submission.',
                 train_seed=TRAIN_SEED,init_seed=INIT_SEED)
write_json(summary_path,summary);emit('RESUME2048_START',summary)
try:
    if len(summary['preflights'])<3:
        for mode in ('head_only','fixed_snake','snake'):
            if any(pr['arm']==mode for pr in summary['preflights']):continue
            path=PARENT/(mode+'_last.pth');cp,config,optimizer,scaler,params=restore_arm(mode,path)
            assert cp['effective_updates']==250
            model.eval();name=sorted(n for n in part if part[n]=='calibration')[0]
            with torch.inference_mode():restored_prediction=candidates(train_dir/name,'native')
            expected=json.loads((PARENT/(mode+'_candidates.json')).read_text())[name]['native']
            assert len(expected)==len(restored_prediction)
            assert all(a['rle']==b['rle'] and a['view']==b['view'] and abs(a['score']-b['score'])<1e-5 for a,b in zip(expected,restored_prediction))
            model.train();model.backbone.eval();item=training_item(schedule[250]);optimizer.zero_grad(set_to_none=True);torch.cuda.reset_peak_memory_stats()
            with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):losses=model([item]);loss=sum(losses.values())
            assert torch.isfinite(loss)
            with torch.enable_grad():scaler.scale(loss).backward()
            scaler.unscale_(optimizer);assert all(p.grad is None or torch.isfinite(p.grad).all() for p in params)
            torch.nn.utils.clip_grad_norm_(params,1.);scaler.step(optimizer);scaler.update();torch.cuda.synchronize()
            probe=dict(arm=mode,status='PASS',restore_step=250,prediction_cache_match=True,
                       next_sample=schedule[250],loss=float(loss.detach()),discarded_optimizer_steps=1,
                       peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3,
                       peak_reserved_gib=torch.cuda.max_memory_reserved()/1024**3)
            summary['preflights'].append(probe);write_json(summary_path,summary);emit('RESUME2048_PREFLIGHT_PASS',probe)
            del item,losses,loss,optimizer,scaler,params,cp;cleanup()
    for target in (600,1200):
        if any(r['target']==target for r in summary['rounds']):continue
        if target==1200:
            round600=next(r for r in summary['rounds'] if r['target']==600)
            improved=any(a['calibration']['PQ']>summary['parent_calibration_pq'][a['arm']]+.001 for a in round600['arms'])
            if not improved:
                summary['status']='stopped_no_calibration_improvement';summary['stop_reason']='No600 arm improved its matched250 PQ by>.001.'
                emit('RESUME2048_EARLY_STOP',dict(reason=summary['stop_reason']));break
            assert time.monotonic()-EXPERIMENT_START+5500<WALL_CAP_SECONDS,'Insufficient time for matched1200 round'
        phase=dict(target=target,arms=[])
        if 'incomplete_round' in summary and summary['incomplete_round']['target']==target:phase=summary['incomplete_round']
        summary['incomplete_round']=phase;summary['status']='training';write_json(summary_path,summary)
        for mode in ('head_only','fixed_snake','snake'):
            if any(a['arm']==mode for a in phase['arms']):continue
            candidate=OUT/(mode+'_last.pth')
            path=candidate if candidate.exists() else PARENT/(mode+'_last.pth')
            cp,config,optimizer,scaler,params=restore_arm(mode,path)
            updates=cp['effective_updates'];attempts=cp['attempts'];history=cp['history'];assert 250<=updates<=target
            emit('RESUME2048_RESTORED',dict(arm=mode,step=updates,target=target,input_checkpoint_sha256=RESUME_INPUT_SHA))
            model.train();model.backbone.eval();arm_started=time.monotonic();torch.cuda.reset_peak_memory_stats()
            while updates<target:
                assert time.monotonic()-EXPERIMENT_START<WALL_CAP_SECONDS-1000,'Training wall cap reached'
                assert attempts<target+100,'Too many AMP retries'
                item=training_item(schedule[updates]);optimizer.zero_grad(set_to_none=True);attempts+=1
                with torch.enable_grad(),torch.autocast('cuda',dtype=torch.float16):losses=model([item]);loss=sum(losses.values())
                assert torch.isfinite(loss),'Nonfinite continuation loss'
                old_scale=scaler.get_scale()
                with torch.enable_grad():scaler.scale(loss).backward()
                scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(params,1.);scaler.step(optimizer);scaler.update()
                accepted=scaler.get_scale()>=old_scale;updates+=int(accepted)
                row=dict(attempt=attempts,effective_update=updates,optimizer_updated=bool(accepted),loss=float(loss.detach()),gradient_norm=float(norm),scale=scaler.get_scale(),**schedule[updates-int(accepted)])
                if mode!='head_only':row['alpha']=float(model.sem_seg_head.snake_mask_adapter.alpha.detach())
                history.append(row);del item,losses,loss
                if attempts%50==0:emit('RESUME2048_TRAIN',dict(arm=mode,target=target,**row,elapsed_seconds=round(time.monotonic()-arm_started)))
                if accepted and updates%50==0:save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history)
            integrity=verify_frozen();train_peak=torch.cuda.max_memory_allocated()/1024**3
            save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history)
            model.zero_grad(set_to_none=True);label=mode+'_step'+str(target);metrics=evaluate_arm(label)
            save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history,metrics)
            shutil.copy2(OUT/(mode+'_last.pth'),OUT/(label+'.pth'))
            write_json(OUT/(label+'_history.json'),history)
            result=dict(arm=mode,config=config,target=target,effective_updates=updates,attempts=attempts,calibration=metrics,
                        integrity=integrity,training_peak_allocated_gib=train_peak,elapsed_seconds=round(time.monotonic()-arm_started))
            phase['arms'].append(result);write_json(summary_path,summary);emit('RESUME2048_ARM_COMPLETE',result)
            del cp,optimizer,scaler,params;cleanup();pack_outputs()
        summary['rounds'].append(phase);summary.pop('incomplete_round',None)
        write_json(summary_path,summary);emit('RESUME2048_ROUND_COMPLETE',phase);pack_outputs()
    if summary['status']!='stopped_no_calibration_improvement':summary['status']='complete'
except Exception as error:
    summary['status']='error';summary['error_type']=type(error).__name__;summary['error']=str(error);traceback.print_exc()
    emit('RESUME2048_ERROR',dict(type=type(error).__name__,message=str(error)))
    if 'history' in globals() and 'optimizer' in globals():save_checkpoint(mode,config,updates,attempts,optimizer,scaler,history)
finally:
    summary['elapsed_seconds']=round(time.monotonic()-EXPERIMENT_START)
    summary['completed_at_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat()
    write_json(summary_path,summary);emit('RESUME2048_COMPLETE',summary)
    model.eval();model.zero_grad(set_to_none=True);gc.collect();torch.cuda.empty_cache();pack_outputs()
    print('RESUME2048_ARCHIVE_READY',str(OUT.with_suffix('.zip')),flush=True)


ASSETS: /kaggle/working/solar_assets
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu128 CUDA: 12.8
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 6.7 MB/s eta 0:00:00
Building Detectron2 CPU extension; log: /kaggle/working/detectron2_install.log
Processing ./solar_m2f_runtime/detectron2
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for detectron2: filename=detectron2-0.6-cp313-cp313-linux_x86_64.whl size=6675511 sha256=8e5dcadf29c2ece16efc65e752a682d8f344720bc1f710e861349c1a67d53349
  Stored in directory: /tmp/pip-ephem-wheel-cache-oiy_nx88/wheels/8e/7f

/kaggle/working/solar_assets/solar_runtime.py:48: UserWarning: Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.
  warnings.warn('Using the PyTorch reference deformable-attention operation; architecture is unchanged, execution is slower.')
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/ops/modules/ms_deform_attn.py:87: SyntaxWarning: invalid escape sequence '\s'
  :param input_flatten               (N, \sum_{l=0}^{L-1} H_l \cdot W_l, C)
/kaggle/working/solar_m2f_runtime/Mask2Former/mask2former/modeling/pixel_decoder/msdeformattn.py:314: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


Runtime ready: 0.6 pytorch_reference
SETUP_SECONDS: 116
RESUME2048_START {"experiment": "native2048_matched_resume250", "source_sha256": "6c8b0f2752762d9909071f07e0d0a0c4e5e5bda60d642c9a104d8b3f1e68b0e1", "origin_step250_sha256": {"head_only": "2666428a467377c8bbaaa4069c99e1a0965e65bc88ad0ba18ae5c9f4237de2a5", "fixed_snake": "0f624da140600ccf94addc983dc5fc2482623df49251ebd601db4d9601d85a1f", "snake": "0bb41ccd6abfb250a6f03c0313651bbfb3752bdac5b49cab5107ddc504ad2d21"}, "parent_schedule_sha256": "36908ae7ad379c2b39d253ad92d0c35ec349ec96051e0f6a5ca46c447c7ef27c", "schedule_sha256": "a834032f46b16af3e1a53ac88f30f1d50087e72f9ebf824686d65f8714f3dbdd", "input_size": 2048, "input_policy": "whole native2048, no crop or downsample", "targets": [600, 1200], "rounds": [], "preflights": [], "status": "restoring", "baseline_calibration_pq": 0.4563696594893772, "baseline_public_score": 0.36, "parent_calibration_pq": {"head_only": 0.45524382357028764, "fixed_snake": 0.4541071485523339, "snake": 0.4555

/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


RESUME2048_PREFLIGHT_PASS {"arm": "head_only", "status": "PASS", "restore_step": 250, "prediction_cache_match": true, "next_sample": {"file_name": "20120213063134Lh.jpeg", "source_image_id": "050202-20120213063134Lh", "hflip": false, "vflip": true}, "loss": 28.16954803466797, "discarded_optimizer_steps": 1, "peak_allocated_gib": 4.391199111938477, "peak_reserved_gib": 5.611328125}
RESUME2048_PREFLIGHT_PASS {"arm": "fixed_snake", "status": "PASS", "restore_step": 250, "prediction_cache_match": true, "next_sample": {"file_name": "20120213063134Lh.jpeg", "source_image_id": "050202-20120213063134Lh", "hflip": false, "vflip": true}, "loss": 28.255023956298828, "discarded_optimizer_steps": 1, "peak_allocated_gib": 4.3996405601501465, "peak_reserved_gib": 5.65625}
RESUME2048_PREFLIGHT_PASS {"arm": "snake", "status": "PASS", "restore_step": 250, "prediction_cache_match": true, "next_sample": {"file_name": "20120213063134Lh.jpeg", "source_image_id": "050202-20120213063134Lh", "hflip": false, "v